# Ensnarlment in real spatial networks

This is to explore the ensnarlment between real spatial networks.
Data source: https://github.com/jocpae/VesselGraph

Prerequisite:
- Package "Plotly" for 3D plots
- Package "kaleido" for saving 3D plots

In [ ]:
import numpy as np
import networkx as nx
import pandas as pd

import matplotlib.pyplot as plt

## Functions

### Visualization

In [ ]:
import plotly
import plotly.graph_objects as go
import plotly.io as pio

In [ ]:
# Plotting utilities
import matplotlib.pyplot as plt
import matplotlib.cm as cm
# import matplotlib.colormaps as cm
import matplotlib.colors as mcolors

# plot all edges with edge colors according to the absolute sums
# Utility: Convert matplotlib colormap to Plotly format
def matplotlib_to_plotly_colorscale(cmap_name='seismic', n_levels=256):
    cmap = plt.colormaps.get_cmap(cmap_name)
    return [
        [i / (n_levels - 1), mcolors.to_hex(cmap(i / (n_levels - 1)))]
        for i in range(n_levels)
    ]

# ----------------------------
# Plotting Utilities
def node_trace(G, pos, color='#3f3f40', size=8, nstart=0, label=True):
    x, y, z = zip(*[pos[n] for n in G.nodes()])
    if label:
        return go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers+text',
            marker=dict(size=size, color=color, opacity=0.8),
            text=[str(n+nstart) for n in G.nodes()],
            textposition='top center',
            hoverinfo='text'
        )
    else:
        return go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=size, color=color, opacity=0.8),
            text=[str(n+nstart) for n in G.nodes()],
            textposition='top center',
            hoverinfo='text'
        )

def edge_trace_constant(G, pos, values, color_scale, vrange=None, width=8):
    """Edge trace with edge colors (from values)."""
    if vrange is not None:
        vmin, vmax = vrange
    else:
        vmin, vmax = np.min(values), np.max(values)

    edge_list = list(G.edges())
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.colormaps.get_cmap(color_scale)

    traces = []
    for i, (u, v) in enumerate(edge_list):
        val = values[i]
        color = mcolors.to_hex(cmap(norm(val)))
        x, y, z = zip(pos[u], pos[v])
        traces.append(go.Scatter3d(
            x=[x[0], x[1], None],
            y=[y[0], y[1], None],
            z=[z[0], z[1], None],
            mode='lines',
            line=dict(color=color, width=width),
            hoverinfo='none'
        ))
    return traces

def dummy_colorbar_trace(vmin, vmax, colorscale, xval=0.8, thickness=10, len=0.4, tk_fz=18):
    return go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode='markers',
        marker=dict(
            size=0.1,
            color=[vmin, vmax],
            colorscale=colorscale,
            cmin=vmin,
            cmax=vmax,
            showscale=True,
            colorbar=dict(
                # title='|Column Sum|',
                titleside='right',
                thickness=20,
                len=len,
                x=xval,
                tickfont=dict(size=tk_fz)
            )
        ),
        showlegend=False,
        hoverinfo='none'
    )

def edge_trace(G, pos, color, width=8):
        edge_x, edge_y, edge_z = [], [], []
        for u, v in G.edges():
            x0, y0, z0 = pos[u]
            x1, y1, z1 = pos[v]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
            edge_z += [z0, z1, None]
        return go.Scatter3d(
            x=edge_x, y=edge_y, z=edge_z,
            mode='lines',
            line=dict(color=color, width=width),
            hoverinfo='none',
            showlegend=False
        )

def plot_networkx_12label(G1, G2, pos1, pos2, row_c='#8a8a8a', col_c='#2159d1',
                          width=10, camera=None, savefig=False, figname='', figsize=[1100,700], scale=2):
    """Plot the two networks with edge color specified
    Params:
    ------
    """

    # ----------------------------
    # Build plot traces

    # Nodes (same for both graphs)
    trace_nodes_G1 = node_trace(G1, pos1)
    trace_nodes_G2 = node_trace(G2, pos2, nstart=len(G1.nodes()))

    # Edges
    trace_edges_G1 = edge_trace(G1, pos1, row_c, width=width)
    trace_edges_G2 = edge_trace(G2, pos2, col_c, width=width)

    # ----------------------------
    # Assemble figure

    fig = go.Figure(data=[
        trace_nodes_G1,
        trace_edges_G1,
        trace_nodes_G2,
        trace_edges_G2
    ])

    axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=False,
        )


    fig.update_layout(
        # title="Graph Edge Visualization with Lam_gli Row/Column Sums",
        scene=dict(
            xaxis=dict(title='x'),
            yaxis=dict(title='y'),
            zaxis=dict(title='z'),
            aspectmode="data",
        ),
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=40),
        height=700
    )

    if camera:
        fig.update_layout(scene_camera=camera)
    if savefig:
        fig.write_image(figname+'.png', width=figsize[0], height=figsize[1], scale=scale)
    fig.show()

In [ ]:
def node_trace_nolabel(G, pos, color='#3f3f40', size=8):
    x, y, z = zip(*[pos[n] for n in G.nodes()])
    return go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=size, color=color, opacity=0.8),
        # text=[str(n+nstart) for n in G.nodes()],
        # textposition='top center',
        # hoverinfo='text'
    )

def plot_networkx_12nolabel(G1, G2, pos1, pos2, row_c='#8a8a8a', col_c='#2159d1', size=8, width=10,
                            camera=None, savefig=False, figname='', figsize=[1100,700],
                            scale=2, ticklabel=False, ax_fz=30):
    """Plot the two networks with edge color specified
    Params:
    ------
    """

    # ----------------------------
    # Build plot traces

    # Nodes (same for both graphs)
    trace_nodes_G1 = node_trace_nolabel(G1, pos1, size=size)
    trace_nodes_G2 = node_trace_nolabel(G2, pos2, size=size)

    # Edges
    trace_edges_G1 = edge_trace(G1, pos1, row_c, width=width)
    trace_edges_G2 = edge_trace(G2, pos2, col_c, width=width)

    # ----------------------------
    # Assemble figure

    fig = go.Figure(data=[
        trace_nodes_G1,
        trace_edges_G1,
        trace_nodes_G2,
        trace_edges_G2
    ])

    axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=False,
        )

    if ticklabel:
        scene_dict = dict(
            xaxis=dict(title=dict(text="x", font=dict(size=ax_fz))),
            yaxis=dict(title=dict(text="y", font=dict(size=ax_fz))),
            zaxis=dict(title=dict(text="z", font=dict(size=ax_fz))),
            aspectmode="data",
        )
    else:
        scene_dict = dict(
            xaxis=dict(title=dict(text="x", font=dict(size=ax_fz)), showticklabels=False),
            yaxis=dict(title=dict(text="y", font=dict(size=ax_fz)), showticklabels=False),
            zaxis=dict(title=dict(text="z", font=dict(size=ax_fz)), showticklabels=False),
            aspectmode="data",
        )
    fig.update_layout(
        # title="Graph Edge Visualization with Lam_gli Row/Column Sums",
        scene=scene_dict,
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=0), # this is to make the figure tight; can add more to "t" if need a title
        height=700
    )
    if camera:
        fig.update_layout(scene_camera=camera)
    if savefig:
        fig.write_image(figname+'.png', width=figsize[0], height=figsize[1], scale=scale)
    fig.show()

In [ ]:
def plot_networkx_color1e(G1, pos1, row_sums,
                         cmap='coolwarm', xval=0.8, ns=2, lw=2):
    """Plot the two networks with edge color specified
    Params:
    ------
    """
    val_max = np.array(row_sums).max()
    val_min = np.array(row_sums).min()
    print("max, min abs value in edge vals:", val_max, val_min)

    # Normalize + convert colormap
    colorscale = matplotlib_to_plotly_colorscale(cmap)

    # ----------------------------
    # Build plot traces

    # Nodes (same for both graphs)
    trace_nodes_G1 = node_trace(G1, pos1, size=ns, label=False)
    # trace_nodes_G2 = node_trace(G2, pos2, size=ns, label=False)

    # Edges
    trace_edges_G1 = edge_trace_constant(G1, pos1, row_sums, cmap, width=10)
    # trace_edges_G2 = edge_trace_constant(G2, pos2, col_sums, cmap, width=10)

    # Colorbar (based on G2 edge values)
    colorbar_trace = dummy_colorbar_trace(val_min, val_max, colorscale, xval=xval)

    # ----------------------------
    # Assemble figure

    fig = go.Figure(data=[
        trace_nodes_G1,
        *trace_edges_G1,
        # trace_nodes_G2,
        # *trace_edges_G2,
        colorbar_trace
    ])

    axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=False,
        )


    fig.update_layout(
        # title="Graph Edge Visualization with Lam_gli Row/Column Sums",
        scene=dict(
            xaxis=dict(title='x'),
            yaxis=dict(title='y'),
            zaxis=dict(title='z'),
            aspectmode="data",
        ),
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=40),
        height=700
    )

    fig.show()

def plot_networkx_colore(G1, G2, pos1, pos2, row_sums, col_sums, cmap='coolwarm', val_range=None, xval=0.8, ns=2, lw=2,
                         camera=None, savefig=False, figname='', figsize=[1100,700], scale=2,
                         ticklabel=False, ax_fz=30, cbar_len=0.4):
    """Plot the two networks with edge color specified
    Params:
    ------
    """
    val_max = np.array([row_sums.max(), col_sums.max()]).max()
    val_min = np.array([row_sums.min(), col_sums.min()]).min()
    print("max, min abs value in 1st and 2nd edge vals:", val_max, val_min)
    if val_range is not None:
        val_min, val_max = val_range
    print("max, min in cmap:", val_max, val_min)

    # Normalize + convert colormap
    colorscale = matplotlib_to_plotly_colorscale(cmap)

    # ----------------------------
    # Build plot traces

    # Nodes (same for both graphs)
    trace_nodes_G1 = node_trace(G1, pos1, size=ns, label=False)
    trace_nodes_G2 = node_trace(G2, pos2, size=ns, label=False)

    # Edges
    trace_edges_G1 = edge_trace_constant(G1, pos1, row_sums, cmap, vrange=[val_min, val_max], width=lw)
    trace_edges_G2 = edge_trace_constant(G2, pos2, col_sums, cmap, vrange=[val_min, val_max], width=lw)

    # Colorbar (based on G2 edge values)
    colorbar_trace = dummy_colorbar_trace(val_min, val_max, colorscale, xval=xval, len=cbar_len)

    # ----------------------------
    # Assemble figure

    fig = go.Figure(data=[
        trace_nodes_G1,
        *trace_edges_G1,
        trace_nodes_G2,
        *trace_edges_G2,
        colorbar_trace
    ])

    axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=False,
        )

    if ticklabel:
        scene_dict = dict(
            xaxis=dict(title=dict(text="x", font=dict(size=ax_fz))),
            yaxis=dict(title=dict(text="y", font=dict(size=ax_fz))),
            zaxis=dict(title=dict(text="z", font=dict(size=ax_fz))),
            aspectmode="data",
        )
    else:
        scene_dict = dict(
            xaxis=dict(title=dict(text="x", font=dict(size=ax_fz)), showticklabels=False),
            yaxis=dict(title=dict(text="y", font=dict(size=ax_fz)), showticklabels=False),
            zaxis=dict(title=dict(text="z", font=dict(size=ax_fz)), showticklabels=False),
            aspectmode="data",
        )

    fig.update_layout(
        # title="Graph Edge Visualization with Lam_gli Row/Column Sums",
        scene=scene_dict,
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=0),
        height=700
    )
    if camera:
        fig.update_layout(scene_camera=camera)
    if savefig:
        fig.write_image(figname+'.png', width=figsize[0], height=figsize[1], scale=scale)
    fig.show()

### Gauss linking integral

In [ ]:
###  compute the Gauss link integral

def Gauss_linking_integral(a, b):
    """
    a:tuple; elements are head and tail
    of line segment, each is a (3,) array representing the xyz coordinate.
    """
    # a0, a1 = a[0],a[1]
    # b0, b1 = b[0],b[1]

    R = np.empty((2, 2), dtype=tuple)
    for i in range(2):
        for j in range(2):
            R[i, j] = a[i] - b[j]

    n = []
    cprod = []

    cprod.append(np.cross(R[0, 0], R[0, 1]))
    cprod.append(np.cross(R[0, 1], R[1, 1]))
    cprod.append(np.cross(R[1, 1], R[1, 0]))
    cprod.append(np.cross(R[1, 0], R[0, 0]))

    for c in cprod:
        n.append(c / (np.linalg.norm(c) + 1e-6))

    area1 = np.arcsin(np.dot(n[0], n[1]))
    area2 = np.arcsin(np.dot(n[1], n[2]))
    area3 = np.arcsin(np.dot(n[2], n[3]))
    area4 = np.arcsin(np.dot(n[3], n[0]))

    sign = np.sign(np.cross(a[1] - a[0], b[1] - b[0]).dot(a[0] - b[0]))
    Area = area1 + area2 + area3 + area4

    return (sign * Area)/(4.*np.pi)

### Signed

In [ ]:
def build_sibigraph(B, labels_p1, labels_p2):
    """Build a signed bipartite graph from a matrix B (one - the other parts).
    Params
    ------
        B, np.ndarray (m x n), signed biadjacency matrix
        label_p1, list, labels for row nodes (set 1)
        label_p2, list, labels for column nodes (set 2)
    Returns
    ------
        G, networkx.Graph - with:
            - 'bipartite' node attribute (0 or 1)
            - 'weight' edge attribute (signed)
    """
    m, n = B.shape
    G = nx.Graph()

    # Add nodes in part 1 (rows)
    for i in range(m):
        node = labels_p1[i]
        G.add_node(node, bipartite=0)

    # Add nodes in part 2 (columns)
    for j in range(n):
        node = labels_p2[j]
        G.add_node(node, bipartite=1)

    # Add edges where B[i,j] ≠ 0
    for i in range(m):
        for j in range(n):
            weight = B[i, j]
            if weight != 0:
                u = labels_p1[i]
                v = labels_p2[j]
                G.add_edge(u, v, weight=weight)

    return G

In [ ]:
def draw_sibigraph(G, weight_attr='weight', part0_c='#8a8a8a', part1_c='#2159d1',
                   figsize=(6, 4), figname='s', tol=1e-4, node_size=200, lw_fac=10., labels=True):
    """Visualize a signed bipartite graph
    """
    # Get bipartite node sets
    part0 = [n for n, d in G.nodes(data=True) if d.get("bipartite", 0) == 0]
    part1 = [n for n, d in G.nodes(data=True) if d.get("bipartite", 0) == 1]

    # Arrange node positions: row 0 at y=1, row 1 at y=0
    pos = {}
    loc_diff = (len(part0) - len(part1))/2.
    for i, node in enumerate(part0):
        pos[node] = (i, 1)
    for i, node in enumerate(part1):
        pos[node] = (i+loc_diff, 0)

    # Separate edges by sign
    pos_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get(weight_attr, 0) >= tol]
    neg_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get(weight_attr, 0) <= -tol]

    # Edge widths
    edge_weights = nx.get_edge_attributes(G, weight_attr)
    pos_widths = [abs(edge_weights[e])*lw_fac for e in pos_edges]
    neg_widths = [abs(edge_weights[e])*lw_fac for e in neg_edges]

    # Draw nodes
    plt.figure(figsize=figsize)
    nx.draw_networkx_nodes(G, pos, nodelist=part0, node_color=part0_c, node_size=node_size)
    nx.draw_networkx_nodes(G, pos, nodelist=part1, node_color=part1_c, node_size=node_size)

    # Draw edges
    nx.draw_networkx_edges(G, pos, edgelist=pos_edges, edge_color='black', width=pos_widths)
    nx.draw_networkx_edges(G, pos, edgelist=neg_edges, edge_color='red', style='dashed', width=neg_widths)

    # Draw labels
    if labels:
        nx.draw_networkx_labels(G, pos, font_size=10)

    # Turn off axis
    plt.axis('off')
    # plt.tight_layout()
    plt.savefig(figname+'-biG.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def draw_sibigraph_nodec(G, nodec, weight_attr='weight', cmap_name='tab10',
                   figsize=(6, 4), figname='s', tol=1e-4, lw_fac=10., labels=True):
    """Visualize a signed bipartite graph with node colors separated
    """
    from matplotlib import colormaps

    # Get bipartite node sets
    part0 = [n for n, d in G.nodes(data=True) if d.get("bipartite", 0) == 0]
    part1 = [n for n, d in G.nodes(data=True) if d.get("bipartite", 0) == 1]

    # Arrange node positions: row 0 at y=1, row 1 at y=0
    pos = {}
    loc_diff = (len(part0) - len(part1))/2.
    for i, node in enumerate(part0):
        pos[node] = (i, 1)
    for i, node in enumerate(part1):
        pos[node] = (i+loc_diff, 0)

    # Seperate node colors
    colors = set(nodec)
    cmap = colormaps[cmap_name]
    color_map = {cid: cmap(i / max(len(colors) - 1, 1)) for i, cid in enumerate(colors)}
    node_colors = [color_map[nodec[n]] for n in range(G.number_of_nodes())]

    # Separate edges by sign
    pos_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get(weight_attr, 0) >= tol]
    neg_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get(weight_attr, 0) <= -tol]

    # Edge widths
    edge_weights = nx.get_edge_attributes(G, weight_attr)
    pos_widths = [abs(edge_weights[e])*lw_fac for e in pos_edges]
    neg_widths = [abs(edge_weights[e])*lw_fac for e in neg_edges]

    # Draw nodes
    plt.figure(figsize=figsize)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=300)

    # Draw edges
    nx.draw_networkx_edges(G, pos, edgelist=pos_edges, edge_color='black', width=pos_widths)
    nx.draw_networkx_edges(G, pos, edgelist=neg_edges, edge_color='red', style='dashed', width=neg_widths)

    # Draw labels
    if labels:
        nx.draw_networkx_labels(G, pos, font_size=10)

    # Turn off axis
    plt.axis('off')
    # plt.tight_layout()
    plt.savefig(figname+'-biG-cluster.png', dpi=300, bbox_inches='tight')
    plt.show()

## Exploratory

### import data sets

In [ ]:
path = 'PATH=TO-DATA'
df_nodes = pd.read_csv(path+"NODE-FILE.csv", sep=';')
df_nodes.head()

In [ ]:
df_nodes.info()

In [ ]:
df_edges = pd.read_csv(path+"EDGE-FILE.csv", sep=';')

In [ ]:
df_edges.head()

In [ ]:
df_edges.info()

### Functions

In [ ]:
def extract_region_subgraph(nodes_df, edges_df, region_name):
    """Extract a subgraph for a specific region.
    """
    region_nodes = nodes_df[nodes_df["region_label"] == region_name]
    region_node_ids = set(region_nodes["id"])

    region_edges = edges_df[
        edges_df["node1id"].isin(region_node_ids) &
        edges_df["node2id"].isin(region_node_ids)
    ]

    return region_nodes, region_edges

def build_region_graphs(region_nodes, region_edges, classify=False):
    """Build region graphs from input nodes and edges
    """
    # Full graph
    G_full = nx.Graph()

    # Add nodes with attributes
    for _, row in region_nodes.iterrows():
        G_full.add_node(
            int(row["id"]),
            pos=(row["pos_x"], row["pos_y"], row["pos_z"]),
            # degree=int(row["degree"]),
            is_border=bool(row["isAtSampleBorder"]),
            region=row["region_label"]
        )

    # Add edges with attributes
    for _, row in region_edges.iterrows():
        G_full.add_edge(
            int(row["node1id"]),
            int(row["node2id"]),
            edge_id=int(row["id"]),
            minRadiusAvg=float(row["minRadiusAvg"]),
            minRadiusStd=float(row["minRadiusStd"]),
            hasNodeAtSampleBorder=bool(row["hasNodeAtSampleBorder"]),
            vessel_class=row["vessel_class"]
        )

    if classify:
        # Subgraphs by vessel class of edges -- !! May not be connected
        G_cap = G_full.edge_subgraph(
            [(u, v) for u, v, d in G_full.edges(data=True)
            if d["vessel_class"] == "capillary"]
        ).copy()

        G_av = G_full.edge_subgraph(
            [(u, v) for u, v, d in G_full.edges(data=True)
            if d["vessel_class"] == "arteriole_venule"]
        ).copy()

        G_art = G_full.edge_subgraph(
            [(u, v) for u, v, d in G_full.edges(data=True)
            if d["vessel_class"] == "artery_vein"]
        ).copy()

        return G_full, G_cap, G_av, G_art
    else:
        return G_full

def graph_diagnostics(G: nx.Graph):
    """Generate graph statistics
    """
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()

    # connected components
    if n_nodes > 0:
        n_components = nx.number_connected_components(G)
        giant_cc_size = len(max(nx.connected_components(G), key=len))
    else:
        n_components = 0
        giant_cc_size = 0

    # border nodes
    border_nodes = sum(
        1 for _, d in G.nodes(data=True)
        if d.get("is_border", False)
    )

    # border edges
    border_edges = sum(
        1 for _, _, d in G.edges(data=True)
        if d.get("hasNodeAtSampleBorder", False)
    )

    return {
        "n_nodes": n_nodes,
        "n_edges": n_edges,
        "n_components": n_components,
        "giant_component_size": giant_cc_size,
        "border_nodes": border_nodes,
        "border_edges": border_edges,
    }

In [ ]:
# Let's find the top right corner subgraph and explore more.
def get_cc(G):
    """Returns a list of connected components (each as a set of node IDs),
    sorted by decreasing size.
    """
    components = list(nx.connected_components(G))
    components_sorted = sorted(components, key=len, reverse=True)
    return components_sorted

In [ ]:
def summarize_cc_topk(G, k=10):
    """Summarize features of top k connected components
    """
    components_sorted = get_cc(G)

    print(f"Total number of connected components: {len(components_sorted)}\n")

    summary = []
    for i, comp in enumerate(components_sorted[:k]):
        comp_size = len(comp)

        # Pick one representative node (first one in the set)
        rep_node = next(iter(comp))
        rep_pos = G.nodes[rep_node].get("pos", None)

        summary.append({
            "rank": i + 1,
            "component_size": comp_size,
            "representative_node": rep_node,
            "representative_position": rep_pos
        })

        print(
            f"Rank {i+1}: size = {comp_size}, "
            f"rep node = {rep_node}, "
            f"rep pos = {rep_pos}"
        )

    return summary

In [ ]:
# signed visualization
# Visualization: selected edges darker and others lighter
def build_edge_traces(G, color, width=2, name=None):
    traces = []

    for u, v in G.edges():
        x0, y0, z0 = G.nodes[u]["pos"]
        x1, y1, z1 = G.nodes[v]["pos"]

        traces.append(
            go.Scatter3d(
                x=[x0, x1],
                y=[y0, y1],
                z=[z0, z1],
                mode="lines",
                line=dict(color=color, width=width),
                hoverinfo="none",
                name=name,
                showlegend=False
            )
        )
    return traces


def build_highlight_traces(G, edge_list, color, width=6, name=None):
    traces = []

    for u, v in edge_list:
        if not G.has_edge(u, v):
            continue

        x0, y0, z0 = G.nodes[u]["pos"]
        x1, y1, z1 = G.nodes[v]["pos"]

        traces.append(
            go.Scatter3d(
                x=[x0, x1],
                y=[y0, y1],
                z=[z0, z1],
                mode="lines",
                line=dict(color=color, width=width),
                hoverinfo="none",
                name=name,
                showlegend=False
            )
        )
    return traces


def plot_networkx_colore_sel(G_all, G1, G2, edge_sel1, edge_sel2,
                             ns=3, lw=5, camera=None, savefig=False, figname='', figsize=[1100,700], scale=2):
    """plot edges with selected ones highlighted
    """
    # plot the nodes
    xs, ys, zs = [], [], []
    for n in G_all.nodes():
        x, y, z = G_all.nodes[n]["pos"]
        xs.append(x)
        ys.append(y)
        zs.append(z)

    node_trace = go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode="markers",
        marker=dict(size=ns, color='#3f3f40'),
        name="Nodes",
        showlegend=True
    )

    # Base subgraphs
    traces_G1 = build_edge_traces(G1, color="lightblue",  width=lw)
    traces_G2 = build_edge_traces(G2, color="pink", width=lw)

    # Emphasized edges
    traces_G1_sel = build_highlight_traces(G1, edge_sel1, color="darkblue", width=lw)
    traces_G2_sel = build_highlight_traces(G2, edge_sel2, color="darkred", width=lw) #[edges2[i] for i in edges_neg[1]]

    fig = go.Figure(data=[node_trace, *traces_G1, *traces_G2, *traces_G1_sel, *traces_G2_sel])

    # axisFormat = dict(
    #                 showbackground=True,
    #                 showticklabels=True,
    #                 autorange=True,
    #                 showgrid=True,
    #         )

    fig.update_layout(
        # title="Overlay of G1 and G2 with Highlighted Edges",
        scene=dict(
            xaxis=dict(title="x", showbackground=True),
            yaxis=dict(title="y", showbackground=True),
            zaxis=dict(title="z", showbackground=True),
            aspectmode="data",
        ),
        showlegend=False,
        margin=dict(l=0, r=0, t=50, b=0),
        height=700
    )

    if camera:
        fig.update_layout(scene_camera=camera)
    if savefig:
        fig.write_image(figname+'.png', width=figsize[0], height=figsize[1], scale=scale)

    fig.show()

In [ ]:
def plot_networkx_2colore_sel(G_all, G1, G2, edge_sel11, edge_sel21, edge_sel12, edge_sel22,
                              colorset = [["lightblue", "pink"],["#636EFA", "magenta"], ["darkblue", "darkred"]],
                             ns=3, lw=5, camera=None, savefig=False, figname='',
                              figsize=[1100,700], scale=2, ticklabel=False, ax_fz=30):
    """plot edges with selected ones highlighted: one strong, one internediate
    """
    # plot the nodes
    xs, ys, zs = [], [], []
    for n in G_all.nodes():
        x, y, z = G_all.nodes[n]["pos"]
        xs.append(x)
        ys.append(y)
        zs.append(z)

    node_trace = go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode="markers",
        marker=dict(size=ns, color='#3f3f40'),
        name="Nodes",
        showlegend=True
    )

    # Base subgraphs
    traces_G1 = build_edge_traces(G1, color=colorset[0][0],  width=lw)
    traces_G2 = build_edge_traces(G2, color=colorset[0][1], width=lw)

    # Emphasized edges - 1
    traces_G1_sel1 = build_highlight_traces(G1, edge_sel11, color=colorset[2][0], width=lw)
    traces_G2_sel1 = build_highlight_traces(G2, edge_sel21, color=colorset[2][1], width=lw) #[edges2[i] for i in edges_neg[1]]

    # Emphasized edges - 2
    traces_G1_sel2 = build_highlight_traces(G1, edge_sel12, color=colorset[1][0], width=lw)
    traces_G2_sel2 = build_highlight_traces(G2, edge_sel22, color=colorset[1][1], width=lw)

    fig = go.Figure(data=[node_trace, *traces_G1, *traces_G2, *traces_G1_sel2,
                          *traces_G2_sel2, *traces_G1_sel1, *traces_G2_sel1])

    # axisFormat = dict(
    #                 showbackground=True,
    #                 showticklabels=True,
    #                 autorange=True,
    #                 showgrid=True,
    #         )

    if ticklabel:
        scene_dict = dict(
            xaxis=dict(title=dict(text="x", font=dict(size=ax_fz))),
            yaxis=dict(title=dict(text="y", font=dict(size=ax_fz))),
            zaxis=dict(title=dict(text="z", font=dict(size=ax_fz))),
            aspectmode="data",
        )
    else:
        scene_dict = dict(
            xaxis=dict(title=dict(text="x", font=dict(size=ax_fz)), showticklabels=False),
            yaxis=dict(title=dict(text="y", font=dict(size=ax_fz)), showticklabels=False),
            zaxis=dict(title=dict(text="z", font=dict(size=ax_fz)), showticklabels=False),
            aspectmode="data",
        )


    fig.update_layout(
        # title="Overlay of G1 and G2 with Highlighted Edges",
        scene=scene_dict,
        showlegend=False,
        margin=dict(l=0, r=0, t=0, b=0),
        height=700
    )

    if camera:
        fig.update_layout(scene_camera=camera)
    if savefig:
        fig.write_image(figname+'.png', width=figsize[0], height=figsize[1], scale=scale)

    fig.show()

In [ ]:
#nodes colored version
def plot_networkx_colorne_sel(G_all, G1, G2, edge_sel1, edge_sel2,
                             ns=3, lw=5, camera=None, savefig=False, figname='', figsize=[1100,700], scale=2):
    """plot edges with selected ones highlighted
    """
    # plot the nodes
    node_traces = []
    for region, color in region_color_map.items():
        df_r = df_nodes_sel2[df_nodes_sel2["region_label"] == region]

        node_traces.append(
            go.Scatter3d(
                x=df_r["pos_x"],
                y=df_r["pos_y"],
                z=df_r["pos_z"],
                mode="markers",
                name=f"Region {region}",
                marker=dict(
                    size=ns,
                    color=color,
                    opacity=0.95,
                ),
                hoverinfo="text",
                text=[
                    f"Node {nid}<br>Region: {region}"
                    for nid in df_r["id"]
                ],
                showlegend=False,
            )
        )

    # Base subgraphs
    traces_G1 = build_edge_traces(G1, color="lightblue",  width=lw)
    traces_G2 = build_edge_traces(G2, color="pink", width=lw)

    # Emphasized edges
    traces_G1_sel = build_highlight_traces(G1, edge_sel1, color="darkblue", width=lw)
    traces_G2_sel = build_highlight_traces(G2, edge_sel2, color="darkred", width=lw) #[edges2[i] for i in edges_neg[1]]

    fig = go.Figure(data=[*node_traces, *traces_G1, *traces_G2, *traces_G1_sel, *traces_G2_sel])

    # axisFormat = dict(
    #                 showbackground=True,
    #                 showticklabels=True,
    #                 autorange=True,
    #                 showgrid=True,
    #         )

    fig.update_layout(
        # title="Overlay of G1 and G2 with Highlighted Edges",
        scene=dict(
            xaxis=dict(title="x", showbackground=True),
            yaxis=dict(title="y", showbackground=True),
            zaxis=dict(title="z", showbackground=True),
            aspectmode="data",
        ),
        showlegend=False,
        margin=dict(l=0, r=0, t=50, b=0),
        height=700
    )

    if camera:
        fig.update_layout(scene_camera=camera)
    if savefig:
        fig.write_image(figname+'.png', width=figsize[0], height=figsize[1], scale=scale)

    fig.show()

In [ ]:
#nodes colored version
def plot_networkx_2colorne_sel(G_all, G1, G2, edge_sel11, edge_sel21, edge_sel12, edge_sel22,
                             ns=3, lw=5, camera=None, savefig=False, figname='', figsize=[1100,700], scale=2):
    """plot edges with selected ones highlighted
    """
    # plot the nodes
    node_traces = []
    for region, color in region_color_map.items():
        df_r = df_nodes_sel2[df_nodes_sel2["region_label"] == region]

        node_traces.append(
            go.Scatter3d(
                x=df_r["pos_x"],
                y=df_r["pos_y"],
                z=df_r["pos_z"],
                mode="markers",
                name=f"Region {region}",
                marker=dict(
                    size=ns,
                    color=color,
                    opacity=0.95,
                ),
                hoverinfo="text",
                text=[
                    f"Node {nid}<br>Region: {region}"
                    for nid in df_r["id"]
                ],
                showlegend=False,
            )
        )

    # Base subgraphs
    traces_G1 = build_edge_traces(G1, color="lightblue",  width=lw)
    traces_G2 = build_edge_traces(G2, color="pink", width=lw)

    # Emphasized edges - 1
    traces_G1_sel1 = build_highlight_traces(G1, edge_sel11, color="darkblue", width=lw)
    traces_G2_sel1 = build_highlight_traces(G2, edge_sel21, color="darkred", width=lw) #[edges2[i] for i in edges_neg[1]]

    # Emphasized edges - 2
    traces_G1_sel2 = build_highlight_traces(G1, edge_sel12, color="#636EFA", width=lw)
    traces_G2_sel2 = build_highlight_traces(G2, edge_sel22, color="magenta", width=lw)

    fig = go.Figure(data=[*node_traces, *traces_G1, *traces_G2, *traces_G1_sel2,
                          *traces_G2_sel2, *traces_G1_sel1, *traces_G2_sel1])

    # axisFormat = dict(
    #                 showbackground=True,
    #                 showticklabels=True,
    #                 autorange=True,
    #                 showgrid=True,
    #         )

    fig.update_layout(
        # title="Overlay of G1 and G2 with Highlighted Edges",
        scene=dict(
            xaxis=dict(title="x", showbackground=True),
            yaxis=dict(title="y", showbackground=True),
            zaxis=dict(title="z", showbackground=True),
            aspectmode="data",
        ),
        showlegend=False,
        margin=dict(l=0, r=0, t=50, b=0),
        height=700
    )

    if camera:
        fig.update_layout(scene_camera=camera)
    if savefig:
        fig.write_image(figname+'.png', width=figsize[0], height=figsize[1], scale=scale)

    fig.show()

### Square slicing

#### Functions

In [ ]:
# select vertices in the range
def select_vertices_range(nodes_df, xlim, ylim, zlim):
    """Select vertices in the range
    """
    region_df = nodes_df[
        (nodes_df["pos_x"] > xlim[0]) & (nodes_df["pos_x"] < xlim[1])
    ]
    region_df = region_df[
        (region_df["pos_y"] > ylim[0]) & (region_df["pos_y"] < ylim[1])
    ]
    region_df = region_df[
        (region_df["pos_z"] > zlim[0]) & (region_df["pos_z"] < zlim[1])
    ]

    region_nodes = region_df["id"].values
    region_sum = np.unique(region_df['region_label'].values, return_counts=True)

    return region_nodes, region_sum

In [ ]:
def plot_networkx_colorne(G1, G2, pos1, pos2, row_sums, col_sums, cmap_max=None, cmap='coolwarm', xval=0.8, ns=2.5, lw=3.5,
                         camera=None, savefig=False, figname='', figsize=[1100,700], scale=2):
    """Plot the two networks with edge color specified
    Params:
    ------
    """
    val_max = np.array([row_sums.max(), col_sums.max()]).max()
    val_min = np.array([row_sums.min(), col_sums.min()]).min()
    print("max, min abs value in 1st and 2nd edge vals:", val_max, val_min)
    if cmap_max:
        val_max = min([cmap_max, val_max])
    print("max, min in cmap:", val_max, val_min)

    # Normalize + convert colormap
    colorscale = matplotlib_to_plotly_colorscale(cmap)

    # ----------------------------
    # Build plot traces

    # Nodes (same for both graphs)
    # plot the nodes
    node_traces = []
    for region, color in region_color_map.items():
        df_r = df_nodes_sel2[df_nodes_sel2["region_label"] == region]

        node_traces.append(
            go.Scatter3d(
                x=df_r["pos_x"],
                y=df_r["pos_y"],
                z=df_r["pos_z"],
                mode="markers",
                name=f"Region {region}",
                marker=dict(
                    size=ns,
                    color=color,
                    opacity=0.95,
                ),
                hoverinfo="text",
                text=[
                    f"Node {nid}<br>Region: {region}"
                    for nid in df_r["id"]
                ],
                showlegend=False,
            )
        )

    # Edges
    trace_edges_G1 = edge_trace_constant(G1, pos1, row_sums, cmap, width=lw)
    trace_edges_G2 = edge_trace_constant(G2, pos2, col_sums, cmap, width=lw)

    # Colorbar (based on G2 edge values)
    colorbar_trace = dummy_colorbar_trace(val_min, val_max, colorscale, xval=xval)

    # ----------------------------
    # Assemble figure

    fig = go.Figure(data=[
        *node_traces,
        *trace_edges_G1,
        *trace_edges_G2,
        colorbar_trace
    ])

    axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=False,
        )


    fig.update_layout(
        # title="Graph Edge Visualization with Lam_gli Row/Column Sums",
        scene=dict(
            xaxis=dict(title='X'),
            yaxis=dict(title='Y'),
            zaxis=dict(title='Z'),
            aspectmode="data",
        ),
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=40),
        height=700
    )
    if camera:
        fig.update_layout(scene_camera=camera)
    if savefig:
        fig.write_image(figname+'.png', width=figsize[0], height=figsize[1], scale=scale)
    fig.show()

In [ ]:
# Find optimal solution in the subspace
def critical_flip_angles(u1, u2, eps=1e-8, deci=8):
    """Compute the distinct angles in [0, pi) at which some entry of a vector in the subspace spanned by the two eigenvectors,
    x(theta) = cos(theta)*u1 + sin(theta)*u2, changes sign.

    Params
    ------
    u1, u2, array (n,), the two eigenvectors (smallest eigenvalues of signed Laplacian).
    eps, float, threshold for ignoring nodes with (u1_i, u2_i) ~ (0,0).

    Returns
    ------
    flip_angles, ndarray (m,), sorted angles in [0, pi) where at least one coordinate crosses zero.
    nodes_deg, ndarray (k,), indices of nodes whose (u1_i, u2_i) are numerically ~0.
    """
    u1 = np.asarray(u1, dtype=float)
    u2 = np.asarray(u2, dtype=float)
    assert u1.shape == u2.shape

    # store nodes whose (u1_i, u2_i) are numerically ~0: extra attention is needed!
    norm2 = u1**2 + u2**2
    mask = norm2 > eps
    nodes_deg = np.where(norm2 <= eps)[0]
    if len(nodes_deg) > 0:
        print("zero values at:", nodes_deg)

    if not np.any(mask):
        # No flips, return empty array
        return np.array([]), nodes_deg

    psi = np.arctan2(u2[mask], u1[mask]) # choosing the quadrant correctly

    # Flip angle theta_i = psi_i + pi/2, reduced to [0, pi)
    theta = psi + 0.5 * np.pi
    theta = np.mod(theta, np.pi)

    # Deduplicate with some tolerance
    theta = np.unique(np.round(theta, decimals=deci)) # Careful: maybe more than one nodes have zero at the same time!

    return theta, nodes_deg


def candidate_mid_angles(flip_angles):
    """Given sorted flip angles in [0, pi), construct representative angles
    in each interval where the sign pattern is constant.

    Returns:
    ------
    numpy.array, representative angles in [0, pi).
    """
    if flip_angles.size == 0:
        # No flips: any theta works; choose 0.
        return np.array([0.0])

    flip_angles = np.sort(flip_angles)
    thetas = []

    # First interval: [0, flip_angles[0])
    thetas.append(0.5 * flip_angles[0])

    # Middle intervals: (flip_i, flip_{i+1})
    for a, b in zip(flip_angles[:-1], flip_angles[1:]):
        thetas.append(0.5 * (a + b))

    # Last interval: (flip_angles[-1], pi)
    thetas.append(0.5 * (flip_angles[-1] + np.pi))

    return np.array(thetas)


def sign_patterns_from_angles(u1, u2, angles):
    """Compute sign patterns s_i(theta) = sign( cos(theta)*u1_i + sin(theta)*u2_i )
    for a list of angles.

    Params
    -----
    u1, u2, array_like (n,)
    angles, array_like (m,)

    Returns
    ------
    S : ndarray, shape (n, m), S[:, j] is the sign pattern for angles[j].
    """
    u1 = np.asarray(u1, dtype=float)
    u2 = np.asarray(u2, dtype=float)
    angles = np.asarray(angles, dtype=float)

    cos_t = np.cos(angles)[None, :]  # shape (1, m)
    sin_t = np.sin(angles)[None, :]
    x = u1[:, None] * cos_t + u2[:, None] * sin_t  # shape (n, m)

    S = np.sign(x)
    # Treat zeros as +1?
    # S[S == 0] = 1
    return S

##### Null model

In [ ]:
## Null model 1: randomize the graph structure (vertices and edges) + no dependence with the radius

from typing import Tuple, Optional, Literal

def generate_spatial_null_model(n_nodes: int, m_edges: int, df_edges_sel: pd.DataFrame,
    xyz_ranges: Tuple[Tuple[float, float], Tuple[float, float], Tuple[float, float]],
    directed: bool = False, seed: Optional[int] = None, node_ids: Optional[np.ndarray] = None,
                                p_dist = True):
    """
    Generate a spatial null model:
      - n_nodes nodes with uniform random (x,y,z) in given ranges
      - m_edges random edges among nodes (no self-loops; no multi-edges) [simplest case]
      - edge attribute 'minRadiusAvg' sampled from empirical distribution in df_edges_sel['minRadiusAvg']

    Parameters
    ----------
    n_nodes : int
        Number of nodes.
    m_edges : int
        Number of edges to generate (will error if too large for simple graph).
    df_edges_sel : pd.DataFrame
        Empirical edges dataframe containing column 'minRadiusAvg'.
    xyz_ranges : ((xmin, xmax),(ymin, ymax),(zmin, zmax))
        Ranges for uniform sampling of coordinates.
    directed : bool
        If True, generate directed edges (u->v). If False, undirected edges.
    seed : int or None
        RNG seed.
    node_ids : array or None
        Optional explicit node IDs. If None, uses 0..n_nodes-1.

    Returns
    -------
    nodes_null_df : pd.DataFrame
        Columns: id, pos_x, pos_y, pos_z
    edges_null_df : pd.DataFrame
        Columns: id, node1id, node2id, minRadiusAvg
    G_null : nx.Graph or nx.DiGraph
        NetworkX graph with node positions and edge radii.
    """
    rng = np.random.default_rng(seed)

    # --- Node IDs ---
    if node_ids is None:
        node_ids = np.arange(n_nodes, dtype=np.int64)
    else:
        node_ids = np.asarray(node_ids)
        if len(node_ids) != n_nodes:
            raise ValueError("node_ids length must equal n_nodes")

    # --- 1) Random locations ---
    (xmin, xmax), (ymin, ymax), (zmin, zmax) = xyz_ranges
    xs = rng.uniform(xmin, xmax, size=n_nodes)
    ys = rng.uniform(ymin, ymax, size=n_nodes)
    zs = rng.uniform(zmin, zmax, size=n_nodes)

    nodes_null_df = pd.DataFrame({
        "id": node_ids,
        "pos_x": xs,
        "pos_y": ys,
        "pos_z": zs,
    })

    # --- 2) Random connections (simple graph, no self-loops, no duplicates) ---
    # Maximum edges possible in simple graphs:
    if directed:
        max_edges = n_nodes * (n_nodes - 1)  # ordered pairs excluding self-loops
    else:
        max_edges = n_nodes * (n_nodes - 1) // 2  # unordered pairs

    if m_edges > max_edges:
        raise ValueError(f"Requested m_edges={m_edges} exceeds max_edges={max_edges} for a simple graph.")

    edges_set = set()
    u_list = []
    v_list = []

    # Efficient batching: oversample candidate pairs and keep uniques until full
    # This avoids O(m) python random draws for large m.
    while len(edges_set) < m_edges:
        remaining = m_edges - len(edges_set)
        # Oversample factor to reduce loops; tuneable
        batch = int(min(max(5 * remaining, 10_000), 2_000_000))

        u = rng.integers(0, n_nodes, size=batch, dtype=np.int64)
        v = rng.integers(0, n_nodes, size=batch, dtype=np.int64)

        mask = (u != v)
        u = u[mask]
        v = v[mask]

        if not directed:
            a = np.minimum(u, v)
            b = np.maximum(u, v)
            pairs = np.stack([a, b], axis=1)
        else:
            pairs = np.stack([u, v], axis=1)

        # sort by distance
        if p_dist:
            # obtain the nodes
            starts = pairs[:, 0]
            ends = pairs[:, 1]

            # Coordinate differences
            dx = xs[starts] - xs[ends]
            dy = ys[starts] - ys[ends]
            dz = zs[starts] - zs[ends]

            # Squared Euclidean distances (avoid sqrt for speed)
            dist2 = dx*dx + dy*dy + dz*dz

            # Sorting indices (ascending distance)
            order = np.argsort(dist2)

            # Sorted pairs -- we can add randomization if needed
            pairs_sorted = pairs[order]
        else:
            pairs_sorted = pairs.copy()

        # Add to set until filled
        for uu, vv in pairs_sorted:
            if len(edges_set) >= m_edges:
                break
            key = (int(uu), int(vv))
            if key not in edges_set:
                edges_set.add(key)
                u_list.append(node_ids[uu])
                v_list.append(node_ids[vv])

    u_arr = np.asarray(u_list, dtype=node_ids.dtype)
    v_arr = np.asarray(v_list, dtype=node_ids.dtype)

    # --- 3) Random radii from empirical distribution ---
    if "minRadiusAvg" not in df_edges_sel.columns:
        raise ValueError("df_edges_sel must contain column 'minRadiusAvg'")

    radii = df_edges_sel["minRadiusAvg"].to_numpy()
    radii = radii[np.isfinite(radii)]
    if radii.size == 0:
        raise ValueError("No finite values found in df_edges_sel['minRadiusAvg'].")

    if m_edges == len(radii):
        print("shuffle the values")
        sampled_radii = rng.permutation(radii)
        # print(sampled_radii)
    else:
        sampled_radii = rng.choice(radii, size=m_edges, replace=True) # choose from the list without replacement

    edges_null_df = pd.DataFrame({
        "id": np.arange(m_edges, dtype=np.int64),
        "node1id": u_arr,
        "node2id": v_arr,
        "minRadiusAvg": sampled_radii,
    })

    # --- Build NetworkX graph (optional but convenient) ---
    G_null = nx.DiGraph() if directed else nx.Graph()

    # Add nodes with positions
    # (using tuples for Plotly + downstream analysis)
    for nid, x, y, z in zip(nodes_null_df["id"].values, xs, ys, zs):
        G_null.add_node(int(nid), pos=(float(x), float(y), float(z)))

    # Add edges with radius attribute
    for uu, vv, rr in zip(edges_null_df["node1id"].values, edges_null_df["node2id"].values, sampled_radii):
        G_null.add_edge(int(uu), int(vv), minRadiusAvg=float(rr))

    return nodes_null_df, edges_null_df, G_null

In [ ]:
## Null model 2: randomize the the radius
def generate_radii_null_model(m_edges: int, df_edges_sel: pd.DataFrame, df_nodes_sel: pd.DataFrame,
                              directed: bool = False, seed: Optional[int] = None):
    """
    Generate a spatial null model:
      - edge attribute 'minRadiusAvg' sampled from empirical distribution in df_edges_sel['minRadiusAvg']

    Parameters
    ----------
    m_edges : int
        Number of edges to generate (will error if too large for simple graph).
    df_edges_sel : pd.DataFrame
        Empirical edges dataframe containing column 'minRadiusAvg'.
    df_nodes_sel : pd.DataFrame
        Nodes nodes dataframe.
    directed : bool
        If True, generate directed edges (u->v). If False, undirected edges.
    seed : int or None
        RNG seed.

    Returns
    -------
    edges_null_df : pd.DataFrame
        Columns: id, node1id, node2id, minRadiusAvg
    G_null : nx.Graph or nx.DiGraph
        NetworkX graph with node positions and edge radii.
    """
    rng = np.random.default_rng(seed)

    # --- shuffle the radius ---
    if "minRadiusAvg" not in df_edges_sel.columns:
        raise ValueError("df_edges_sel must contain column 'minRadiusAvg'")

    radii = df_edges_sel["minRadiusAvg"].to_numpy()
    radii = radii[np.isfinite(radii)]
    if radii.size == 0:
        raise ValueError("No finite values found in df_edges_sel['minRadiusAvg'].")

    if m_edges == len(radii):
        print("Shuffle the values")
        sampled_radii = rng.permutation(radii)
        # print(sampled_radii)
    else:
        print("Sample {} values from {} radii".format(m_edges, len(radii)))
        sampled_radii = rng.choice(radii, size=m_edges, replace=True) # choose from the list without replacement

    edges_null_df =  df_edges_sel.copy()
    edges_null_df["minRadiusAvg"] = sampled_radii

    # --- Build NetworkX graph (optional but convenient) ---
    G_null = nx.DiGraph() if directed else nx.Graph()

    # Add nodes with positions
    # (using tuples for Plotly + downstream analysis)
    for nid, x, y, z in zip(df_nodes_sel["id"].values,
                            df_nodes_sel["pos_x"].values, df_nodes_sel["pos_y"].values, df_nodes_sel["pos_z"].values):
        G_null.add_node(int(nid), pos=(float(x), float(y), float(z)))

    # Add edges with radius attribute
    for uu, vv, rr in zip(edges_null_df["node1id"].values, edges_null_df["node2id"].values, sampled_radii):
        G_null.add_edge(int(uu), int(vv), minRadiusAvg=float(rr))

    return edges_null_df, G_null

#### Zone I: cerebellum

In [ ]:
# --- Step 1: select a region ---
xlim = [1055., 1120.]
ylim = [3160., 3260.]
zlim = [855., 1085.]

In [ ]:
nodes_sel, regs_sum = select_vertices_range(df_nodes, xlim, ylim, zlim)
print("regions:", regs_sum[0])
print("counts:", regs_sum[1])

In [ ]:
# --- Step 2: select edges touching at least one selected node ---
df_edges_sel = df_edges[df_edges["node1id"].isin(nodes_sel) | df_edges["node2id"].isin(nodes_sel)]

# --- Step 3: update node set with all incident nodes ---
nodes_sel2 = set(df_edges_sel["node1id"]).union(set(df_edges_sel["node2id"]))
df_nodes_sel2 = df_nodes[df_nodes["id"].isin(nodes_sel2)]

print(f"Initial selected nodes: {len(nodes_sel)}")
print(f"Selected edges: {len(df_edges_sel)}")
print(f"Updated node set: {len(nodes_sel2)}")

In [ ]:
# create a node label dictionary
n2i_dict = {id:i for i,id in enumerate(df_nodes_sel2['id'].values)}
print("#nodes in dictionary", len(n2i_dict))

In [ ]:
# check the regions
np.unique(df_nodes_sel2['region_label'].values, return_counts=True)

In [ ]:
## Plot
# Region labels and colors
region_categories = sorted(regs_sum[0])
print(region_categories)

region_color_map = {
    region_categories[0]: '#BCBD22',
    region_categories[1]: '#9467BD',
    region_categories[2]: '#F58518', #'#8C564B'
    region_categories[3]: '#FF7F0E', # orange
    region_categories[4]: '#109618', # green
    region_categories[5]: '#620042', # dark brown
}

In [ ]:
# Build a fast lookup for node positions - edges color ~ radius
ns = 2.5
lw = 3.5
xval = 0.8
cmap = 'coolwarm'

pos_dict = df_nodes_sel2.set_index("id")[
    ["pos_x", "pos_y", "pos_z"]
].to_dict("index")

# prepare the color map
r_vals = df_edges_sel["minRadiusAvg"].values
val_min, val_max = r_vals.min(), r_vals.max()
print("max, min abs value in edge vals:", val_max, val_min)

# Normalize + convert colormap
colorscale = matplotlib_to_plotly_colorscale(cmap)
# Colorbar
colorbar_trace = dummy_colorbar_trace(val_min, val_max, colorscale, xval=xval)

# assign colors
norm = mcolors.Normalize(vmin=val_min, vmax=val_max)
cmap_vals = plt.colormaps.get_cmap(cmap)

edge_traces = []
for _, edge in df_edges_sel.iterrows():
    n1, n2 = edge["node1id"], edge["node2id"]

    if (n1 not in pos_dict) or (n2 not in pos_dict):
        continue

    x0, y0, z0 = pos_dict[n1]["pos_x"], pos_dict[n1]["pos_y"], pos_dict[n1]["pos_z"]
    x1, y1, z1 = pos_dict[n2]["pos_x"], pos_dict[n2]["pos_y"], pos_dict[n2]["pos_z"]

    r = edge["minRadiusAvg"]
    # normalized_r = (r - r_min) / (r_max - r_min + 1e-12)
    color = mcolors.to_hex(cmap_vals(norm(r)))
    edge_traces.append(go.Scatter3d(
            x=[x0, x1, None],
            y=[y0, y1, None],
            z=[z0, z1, None],
            mode="lines",
            line=dict(color=color, width=lw),
            hoverinfo="text",
            text=f"Radius: {r:.3f}",
            showlegend=False,
        ))

node_traces = []
for region, color in region_color_map.items():
    df_r = df_nodes_sel2[df_nodes_sel2["region_label"] == region]

    node_traces.append(
        go.Scatter3d(
            x=df_r["pos_x"],
            y=df_r["pos_y"],
            z=df_r["pos_z"],
            mode="markers",
            name=f"Region {region}",
            marker=dict(
                size=ns,
                color=color,
                opacity=0.9,
            ),
            hoverinfo="text",
            text=[
                f"Node {nid}<br>Region: {region}"
                for nid in df_r["id"]
            ],
        )
    )

fig = go.Figure(data=[*node_traces, *edge_traces, colorbar_trace])

axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=True,
        )


fig.update_layout(
    # title="3D Brain Vascular Subnetwork",
    scene=dict(
        xaxis=dict(title="x"),
        yaxis=dict(title="y"),
        zaxis=dict(title="z"),
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    # legend=dict(itemsizing="constant"),
)

fig.show()

In [ ]:
#set an appropriate camera (for later plots as well)
camera = dict(
    eye=dict(x=1.8, y=1.1, z=.5),
    center=dict(x=0, y=0, z=0),
    up=dict(x=0, y=0, z=.1)
)
fig.update_layout(scene_camera=camera)
fig.show()

In [ ]:
pio.write_image(fig, "vessel-z1_view.png", width=700, height=600, scale=2) #, scale=2

In [ ]:
# output the range
full_fig = fig.full_figure_for_development()
print(full_fig.layout.scene.xaxis.range)
print(full_fig.layout.scene.yaxis.range)
print(full_fig.layout.scene.zaxis.range)

##### Edge counts

In [ ]:
### see how #edges change along the radius cuts
col_rad = "minRadiusAvg"
# cuts = np.arange(1., 16., 0.5)
cuts = sorted(df_edges_sel[col_rad].unique()[1:-1])

summary = []
for cut in cuts:
    # summarize the edges below and above
    num_below = len(df_edges_sel[df_edges_sel[col_rad] < cut])
    num_above = len(df_edges_sel[df_edges_sel[col_rad] >= cut])

    summary.append([cut, num_below, num_above])

In [ ]:
# visualize the summary
tk_fz = 14
lb_fz = 16
plt.figure(figsize=(7,3.5))
summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
# plt.legend()
# plt.grid()
plt.savefig("vessel-z1-edgrad.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# visualize the summary + cuts
cut_sel = [1.5, 2, 5]

plt.figure(figsize=(7,3.5))
# plot vertical lines
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
# plt.legend()
# plt.grid()
plt.savefig("vessels-z1-edgradc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Q: how does the edge distance correlate with radius?
res_dist = []
res_radi= []
G2 = build_region_graphs(df_nodes_sel2, df_edges_sel)

for _, row in df_edges_sel.iterrows():
    n1, n2 = row["node1id"], row["node2id"]
    res_radi.append(float(row["minRadiusAvg"]))

    # compute the distance between the two nodes
    pos1 = np.array(G2.nodes[n1]['pos'])
    pos2 = np.array(G2.nodes[n2]['pos'])
    dist = np.linalg.norm(pos1-pos2)
    res_dist.append(dist)

In [ ]:
# pearson correlation
from scipy import stats

corr = stats.pearsonr(res_dist, res_radi)
print("correlation:", corr)

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels2-dist-radius.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels2-dist-radius-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Unbalance scores + Centralities

In [ ]:
### See how balance changes with cut
# sort edges
col_rad = "minRadiusAvg"
df_edges_sel = df_edges_sel.sort_values(by=col_rad)

G_sel = build_region_graphs(df_nodes_sel2, df_edges_sel)

## compute the full GLI between each pair of edges in order
Lgli_full = np.zeros((len(df_edges_sel), len(df_edges_sel)))
for i in range(Lgli_full.shape[0]):
    e1 = [df_edges_sel.iloc[i]['node1id'], df_edges_sel.iloc[i]['node2id']]
    e1_pos = [np.array(G_sel.nodes[e1[0]]['pos']), np.array(G_sel.nodes[e1[1]]['pos'])]
    for j in range(i+1, Lgli_full.shape[1]):
        e2 = [df_edges_sel.iloc[j]['node1id'], df_edges_sel.iloc[j]['node2id']]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lgli_full[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G_sel.nodes[e2[0]]['pos']), np.array(G_sel.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lgli_full[i, j] = gli_ij

In [ ]:
### See how *centrality distribution* changes with cut
res_cut = []

res_bal = []
res_cme = []
res_csd = []
res_cax = []
idx = 0
for _, row in df_edges_sel.iterrows():

    # obtain the linking
    Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
    print("Lam_gli shape:", Lam_gli.shape)

    if Lam_gli.shape[1] == 0:
        continue

    # record the cut
    res_cut.append(float(row[col_rad]))
    print("cut:", res_cut[-1])

    idx += 1

    # weigh matrix of the bipartite graph
    W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                      [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

    ## Balance score
    # prepare connected components if any
    Gb = nx.from_numpy_array(W_gli)
    Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
    print("#connected components:", len(Gb_ccs))
    print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

    # for each cc
    res = []
    for idx_g in range(len(Gb_ccs)):
        Gb_cc = Gb_ccs[idx_g]
        if Gb_cc.number_of_nodes() < 3:
            continue
        # get the adjacency
        W_cc = nx.to_numpy_array(Gb_cc)
        deg = np.sum(abs(W_cc), axis=1)

        # symmetric version
        D2_inv = np.diag(1./np.sqrt(deg))
        P_sym = D2_inv.dot(W_cc).dot(D2_inv)
        eigvals = np.linalg.eigvals(P_sym)
        # print("Max eigenval of P:", np.max(eigvals))
        res.append(np.max(eigvals))
    print("balance score:", res)
    # set one balance score for each cut value
    if len(res) == 0:
        res.append(np.nan)
    else:
        res_bal.append(np.mean(res).real)

    ## Centralities
    res = np.sum(abs(W_gli), axis=0)
    res_cme.append(np.mean(res))
    res_csd.append(np.std(res))
    res_cax.append(np.max(res))

1. Unbalance score

In [ ]:
res_cut = np.array(res_cut)
res_bal = np.array(res_bal)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'bal': res_bal})
df_res.to_csv("vessels-z1-balrad.csv", index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-balrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unalance score')
plt.savefig("vessels-z1-unbalrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.savefig("vessels-z1-unbalrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# set one balance score for each cut value
res_cut = np.array(res_cut)
res_cme = np.array(res_cme)
res_csd = np.array(res_csd)
res_cax = np.array(res_cax)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'cent-mean': res_cme, 'cent-std': res_csd, 'cent-max': res_cax})
df_res.to_csv("vessels-z1-cenrad.csv", index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cenrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cenrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cmxrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cmxrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

###### Null model 1: SER and SRG

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['er', 'dist'][1]
p_dist = True if lab_mod == 'dist' else False
print("probability by distance?", p_dist)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    nodes_null_df, edges_null_df, G_null = generate_spatial_null_model(
        n_nodes=num_nodes, m_edges=num_edges, df_edges_sel=df_edges_sel,
        xyz_ranges=xyz_ranges, directed=False, p_dist=p_dist)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij
    # compute the balance score
    res_cut = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # characterize the balance
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.savefig("vessels-z1-unbalrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-unbalrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z1-unbalrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-unbalrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
  # plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cenrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.savefig("vessels-z1-cmerad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-cmerad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.savefig("vessels-z1-caxrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
#  plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-caxrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z1-cenrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

###### Null model 2: shuffle radius

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['radi'][0]
print("null model", lab_mod)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cut = []
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    edges_null_df, G_null = generate_radii_null_model(
        m_edges=num_edges, df_edges_sel=df_edges_sel, df_nodes_sel=df_nodes_sel2, directed=False)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij

    # compute the balance score
    res_cut = []
    res_bal = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # weight matrix of the bipartite graph
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        ## Balance
        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
plt.savefig("vessels-z1-unbalrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-unbalrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z1-unbalrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-unbalrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_bal = pd.read_csv("vessels-z1-balrad.csv")
df_radi = pd.read_csv("vessels-z1-unbalrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.legend()
plt.savefig("vessels-z1-unbalrad-null2.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-unbalrad-2.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
# plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cenrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-cmerad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-cmerad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z1-caxrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
#  plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-caxrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z1-cenrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_cen = pd.read_csv("vessels-z1-cenrad.csv")
df_radi = pd.read_csv("vessels-z1-cenrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.legend()
plt.savefig("vessels-z1-cmerad-null2.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-cmerad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.legend()
plt.savefig("vessels-z1-caxrad-null2.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-caxrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

###### Summary

In [ ]:
# uniform scale
ylim_unb = [-0.05, 0.63]
ylim_cme = [0.0005, 45]
ylim_cax = [0.1, 180]

1. Unbalance scores

In [ ]:
## ALL
df_bal = pd.read_csv("vessels-z1-balrad.csv")
df_radi = pd.read_csv("vessels-z1-unbalrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z1-unbalrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z1-unbalrad-sw-dist-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
tk_fz = 14
lb_fz = 16
lg_fz = 12
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
# plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-unbalrad-all.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-unbalrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_unb)
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-unbalrad-all3.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-unbalrad-all3.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-unbalrad-null3.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-unbalrad-null3.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# read in data
df_cen = pd.read_csv("vessels-z1-cenrad.csv")
df_radi = pd.read_csv("vessels-z1-cenrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z1-cenrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z1-cenrad-sw-dist-s10.csv")

In [ ]:
colors = ['orangered', 'b', 'y', 'c']

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-cmerad-all.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-cmerad-all-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cme)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-cmerad-all3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-cmerad-null3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 1-df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-caxrad-all.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-caxrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-caxrad-all-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-caxrad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cax)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-caxrad-all3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-caxrad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z1-caxrad-null3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z1-caxrad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Single cut

In [ ]:
# Is the graph connected?
G_sel2 = build_region_graphs(df_nodes_sel2, df_edges_sel)
component_summary = summarize_cc_topk(G_sel2, k=10)

###### cut = 2

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 2

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z1-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [550, 700]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'
# Set tick positions (match number of labels)
# xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
# yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
# plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
# plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.008588
cmax = 2.0399
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [800, 700]

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 1000 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
## select the colors
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

###### cut = 5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z1-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [550, 700]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'
# Set tick positions (match number of labels)
# xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
# yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
# plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
# plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.008588
cmax = 2.0399
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [800, 700]

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 0.01
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

Bi-clustering 1: choose the smallest eigenvalue:

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, # colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-1')

Bi-clustering 2: combine smallest eigenvectors

In [ ]:
# Combine the first two eigenvectors to see whether we have better results?
tol = 1e-4

from itertools import combinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol_ev = 0.01
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol_ev]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("#Flip angles:", len(flip_angles))
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = []
best_partition = []

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost-tol:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif abs(cost-best_cost) < tol:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost/fac)
print("# Best partitions:", len(best_partition))

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-2')

###### cut = 1.5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 1.5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z1-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [550, 700]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'
# Set tick positions (match number of labels)
# xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
# yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
# plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
# plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.008588
cmax = 2.0399
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [800, 700]

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

#### Zone II: mixed (Midbrain)

In [ ]:
# --- Step 1: select a region ---
xlim = [1026., 1091.]
ylim = [2785., 2885.]
zlim = [945., 1175.]

In [ ]:
nodes_sel, regs_sum = select_vertices_range(df_nodes, xlim, ylim, zlim)
print("regions:", regs_sum[0])
print("counts:", regs_sum[1])

In [ ]:
# --- Step 2: select edges touching at least one selected node ---
df_edges_sel = df_edges[df_edges["node1id"].isin(nodes_sel) | df_edges["node2id"].isin(nodes_sel)]

# --- Step 3: update node set with all incident nodes ---
nodes_sel2 = set(df_edges_sel["node1id"]).union(set(df_edges_sel["node2id"]))
df_nodes_sel2 = df_nodes[df_nodes["id"].isin(nodes_sel2)]

print(f"Initial selected nodes: {len(nodes_sel)}")
print(f"Selected edges: {len(df_edges_sel)}")
print(f"Updated node set: {len(nodes_sel2)}")

In [ ]:
# create a node label dictionary
n2i_dict = {id:i for i,id in enumerate(df_nodes_sel2['id'].values)}
print("#nodes in dictionary", len(n2i_dict))

In [ ]:
# check the regions
np.unique(df_nodes_sel2['region_label'].values, return_counts=True)

In [ ]:
# Region labels and colors
region_categories = np.unique(df_nodes_sel2['region_label'].values)
print(region_categories)

# region_color_map = {
#     region_categories[0]: "red",
#     region_categories[1]: "blue",
#     region_categories[2]: "black",
# }
region_color_map = {
    region_categories[0]: '#7F7F7F',
    region_categories[1]: '#DD4477',
    region_categories[2]: '#7F7F7F', # grey /'#8C564B'
    region_categories[3]: '#FF7F0E', # orange
    region_categories[4]: '#109618', # green
    region_categories[5]: '#620042', # dark brown
    region_categories[6]: '#9467BD',
    region_categories[7]: '#F58518',
    region_categories[8]: '#BCBD22', # dark brown
}

In [ ]:
# Build a fast lookup for node positions - edges color ~ radius
ns = 2.5
lw = 3.5
xval = 0.8
cmap = 'coolwarm'

pos_dict = df_nodes_sel2.set_index("id")[
    ["pos_x", "pos_y", "pos_z"]
].to_dict("index")

# prepare the color map
r_vals = df_edges_sel["minRadiusAvg"].values
val_min, val_max = r_vals.min(), r_vals.max()
print("max, min abs value in edge vals:", val_max, val_min)

# Normalize + convert colormap
colorscale = matplotlib_to_plotly_colorscale(cmap)
# Colorbar
colorbar_trace = dummy_colorbar_trace(val_min, val_max, colorscale, xval=xval)

# assign colors
norm = mcolors.Normalize(vmin=val_min, vmax=val_max)
cmap_vals = plt.colormaps.get_cmap(cmap)

edge_traces = []
for _, edge in df_edges_sel.iterrows():
    n1, n2 = edge["node1id"], edge["node2id"]

    if (n1 not in pos_dict) or (n2 not in pos_dict):
        continue

    x0, y0, z0 = pos_dict[n1]["pos_x"], pos_dict[n1]["pos_y"], pos_dict[n1]["pos_z"]
    x1, y1, z1 = pos_dict[n2]["pos_x"], pos_dict[n2]["pos_y"], pos_dict[n2]["pos_z"]

    r = edge["minRadiusAvg"]
    # normalized_r = (r - r_min) / (r_max - r_min + 1e-12)
    color = mcolors.to_hex(cmap_vals(norm(r)))
    edge_traces.append(go.Scatter3d(
            x=[x0, x1, None],
            y=[y0, y1, None],
            z=[z0, z1, None],
            mode="lines",
            line=dict(color=color, width=lw),
            hoverinfo="text",
            text=f"Radius: {r:.3f}",
            showlegend=False,
        ))

node_traces = []
for region, color in region_color_map.items():
    df_r = df_nodes_sel2[df_nodes_sel2["region_label"] == region]

    node_traces.append(
        go.Scatter3d(
            x=df_r["pos_x"],
            y=df_r["pos_y"],
            z=df_r["pos_z"],
            mode="markers",
            name=f"Region {region}",
            marker=dict(
                size=ns,
                color=color,
                opacity=0.9,
            ),
            hoverinfo="text",
            text=[
                f"Node {nid}<br>Region: {region}"
                for nid in df_r["id"]
            ],
        )
    )

fig = go.Figure(data=[*node_traces, *edge_traces, colorbar_trace])

axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=True,
        )


fig.update_layout(
    # title="3D Brain Vascular Subnetwork",
    scene=dict(
        xaxis=dict(title="x"),
        yaxis=dict(title="y"),
        zaxis=dict(title="z"),
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    # legend=dict(itemsizing="constant"),
)

fig.show()

In [ ]:
#set an appropriate camera (for later plots as well)
camera = dict(
    eye=dict(x=1.63, y=1., z=.6), # eye=dict(x=1.8, y=1.1, z=.5),
    center=dict(x=0, y=0, z=0),
    up=dict(x=0, y=0, z=.1)
)
fig.update_layout(scene_camera=camera)
fig.show()

In [ ]:
pio.write_image(fig, "vessel-z2_view.png", width=700, height=600, scale=2) #, scale=2

In [ ]:
# output the range
full_fig = fig.full_figure_for_development()
print(full_fig.layout.scene.xaxis.range)
print(full_fig.layout.scene.yaxis.range)
print(full_fig.layout.scene.zaxis.range)

##### Edge counts

In [ ]:
### see how #edges change along the radius cuts
col_rad = "minRadiusAvg"
# cuts = np.arange(1., 16., 0.5)
cuts = sorted(df_edges_sel[col_rad].unique()[1:-1])

summary = []
for cut in cuts:
    # summarize the edges below and above
    num_below = len(df_edges_sel[df_edges_sel[col_rad] < cut])
    num_above = len(df_edges_sel[df_edges_sel[col_rad] >= cut])

    summary.append([cut, num_below, num_above])

In [ ]:
# visualize the summary
tk_fz = 14
lb_fz = 16
plt.figure(figsize=(7,3.5))
summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.savefig("vessel-z2-edgrad.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# visualize the summary + cuts
cut_sel = [1.5, 2.5, 5]

plt.figure(figsize=(7,3.5))
# plot vertical lines
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.savefig("vessels-z2-edgradc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Q: how does the edge distance correlate with radius?
res_dist = []
res_radi= []
G2 = build_region_graphs(df_nodes_sel2, df_edges_sel)

for _, row in df_edges_sel.iterrows():
    n1, n2 = row["node1id"], row["node2id"]
    res_radi.append(float(row["minRadiusAvg"]))

    # compute the distance between the two nodes
    pos1 = np.array(G2.nodes[n1]['pos'])
    pos2 = np.array(G2.nodes[n2]['pos'])
    dist = np.linalg.norm(pos1-pos2)
    res_dist.append(dist)

In [ ]:
# pearson correlation
from scipy import stats

corr = stats.pearsonr(res_dist, res_radi)
print("correlation:", corr)

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels2-dist-radius.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels2-dist-radius-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Unbalance scores + Centralities

In [ ]:
### See how balance changes with cut
# sort edges
col_rad = "minRadiusAvg"
df_edges_sel = df_edges_sel.sort_values(by=col_rad)

G_sel = build_region_graphs(df_nodes_sel2, df_edges_sel)

## compute the full GLI between each pair of edges in order
Lgli_full = np.zeros((len(df_edges_sel), len(df_edges_sel)))
for i in range(Lgli_full.shape[0]):
    e1 = [df_edges_sel.iloc[i]['node1id'], df_edges_sel.iloc[i]['node2id']]
    e1_pos = [np.array(G_sel.nodes[e1[0]]['pos']), np.array(G_sel.nodes[e1[1]]['pos'])]
    for j in range(i+1, Lgli_full.shape[1]):
        e2 = [df_edges_sel.iloc[j]['node1id'], df_edges_sel.iloc[j]['node2id']]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lgli_full[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G_sel.nodes[e2[0]]['pos']), np.array(G_sel.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lgli_full[i, j] = gli_ij

In [ ]:
### See how *centrality distribution* changes with cut
res_cut = []

res_bal = []
res_cme = []
res_csd = []
res_cax = []
idx = 0
for _, row in df_edges_sel.iterrows():

    # obtain the linking
    Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
    print("Lam_gli shape:", Lam_gli.shape)

    if Lam_gli.shape[1] == 0:
        continue

    # record the cut
    res_cut.append(float(row[col_rad]))
    print("cut:", res_cut[-1])

    idx += 1

    # weigh matrix of the bipartite graph
    W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                      [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

    ## Balance score
    # prepare connected components if any
    Gb = nx.from_numpy_array(W_gli)
    Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
    print("#connected components:", len(Gb_ccs))
    print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

    # for each cc
    res = []
    for idx_g in range(len(Gb_ccs)):
        Gb_cc = Gb_ccs[idx_g]
        if Gb_cc.number_of_nodes() < 3:
            continue
        # get the adjacency
        W_cc = nx.to_numpy_array(Gb_cc)
        deg = np.sum(abs(W_cc), axis=1)

        # symmetric version
        D2_inv = np.diag(1./np.sqrt(deg))
        P_sym = D2_inv.dot(W_cc).dot(D2_inv)
        eigvals = np.linalg.eigvals(P_sym)
        # print("Max eigenval of P:", np.max(eigvals))
        res.append(np.max(eigvals))
    print("balance score:", res)
    # set one balance score for each cut value
    if len(res) == 0:
        res.append(np.nan)
    else:
        res_bal.append(np.mean(res).real)

    ## Centralities
    res = np.sum(abs(W_gli), axis=0)
    res_cme.append(np.mean(res))
    res_csd.append(np.std(res))
    res_cax.append(np.max(res))

1. Unbalance score

In [ ]:
res_cut = np.array(res_cut)
res_bal = np.array(res_bal)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'bal': res_bal})
df_res.to_csv("vessels-z2-balrad.csv", index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-balrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unalance score')
plt.savefig("vessels-z2-unbalrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.savefig("vessels-z2-unbalrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# set one balance score for each cut value
res_cut = np.array(res_cut)
res_cme = np.array(res_cme)
res_csd = np.array(res_csd)
res_cax = np.array(res_cax)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'cent-mean': res_cme, 'cent-std': res_csd, 'cent-max': res_cax})
df_res.to_csv("vessels-z2-cenrad.csv", index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z2-cenrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cenrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cenrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cmxrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cmxrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

###### Null model 1: SER and SRG

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['er', 'dist'][1]
p_dist = True if lab_mod == 'dist' else False
print("probability by distance?", p_dist)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    nodes_null_df, edges_null_df, G_null = generate_spatial_null_model(
        n_nodes=num_nodes, m_edges=num_edges, df_edges_sel=df_edges_sel,
        xyz_ranges=xyz_ranges, directed=False, p_dist=p_dist)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij
    # compute the balance score
    res_cut = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # characterize the balance
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
plt.savefig("vessels-z2-unbalrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-unbalrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z2-unbalrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z2-unbalrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
  # plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cenrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-cenrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cmerad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-cmerad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-caxrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
#  plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-caxrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z2-cenrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z2-cenrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

###### Null model 2: shuffle radius

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['radi'][0]
print("null model", lab_mod)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cut = []
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    edges_null_df, G_null = generate_radii_null_model(
        m_edges=num_edges, df_edges_sel=df_edges_sel, df_nodes_sel=df_nodes_sel2, directed=False)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij

    # compute the balance score
    res_cut = []
    res_bal = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # weight matrix of the bipartite graph
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        ## Balance
        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
plt.savefig("vessels-z2-unbalrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-unbalrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z2-unbalrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z2-unbalrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_bal = pd.read_csv("vessels-z2-balrad.csv")
df_radi = pd.read_csv("vessels-z2-unbalrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.legend()
plt.savefig("vessels-z2-unbalrad-null2.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-unbalrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
# plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z2-cenrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-cenrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.savefig("vessels-z2-cmerad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-cmerad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.savefig("vessels-z2-caxrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
#  plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-caxrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z2-cenrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z2-cenrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_cen = pd.read_csv("vessels-z2-cenrad.csv")
df_radi = pd.read_csv("vessels-z2-cenrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.legend()
plt.savefig("vessels-z2-cmerad-null2.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-cmerad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.legend()
plt.savefig("vessels-z2-caxrad-null2.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-caxrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

###### Summary

1. Unbalance scores

In [ ]:
## ALL
df_bal = pd.read_csv("vessels-z2-balrad.csv")
df_radi = pd.read_csv("vessels-z2-unbalrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z2-unbalrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z2-unbalrad-sw-dist-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
tk_fz = 14
lb_fz = 16
lg_fz = 12

plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-unbalrad-all.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-unbalrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_unb)
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-unbalrad-all3.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-unbalrad-all3.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])


# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-unbalrad-null3.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-unbalrad-null3.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# read in data
df_cen = pd.read_csv("vessels-z2-cenrad.csv")
df_radi = pd.read_csv("vessels-z2-cenrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z2-cenrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z2-cenrad-sw-dist-s10.csv")

In [ ]:
colors = ['orangered', 'b', 'y', 'c']

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-cmerad-all.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-cenrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-cmerad-all-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-cenrad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cme)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-cmerad-all3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-cenrad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-cmerad-null3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-cenrad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-caxrad-all.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-caxrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-caxrad-all-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-caxrad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cax)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False, loc='lower center')
plt.savefig("vessels-z2-caxrad-all3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-caxrad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z2-caxrad-null3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z2-caxrad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Single cut

In [ ]:
# Is the graph connected?
G_sel2 = build_region_graphs(df_nodes_sel2, df_edges_sel)
component_summary = summarize_cc_topk(G_sel2, k=10)

In [ ]:
figsize_g = [500, 550]
figsize_c = [500, 550]

###### cut = 2.5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 2.5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z2-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [500, 550]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'
# Set tick positions (match number of labels)
# xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
# yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
# plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
# plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.00112
cmax = 2.1692
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [500, 550]

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

Bi-clustering 1: choose the smallest eigenvalue:

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
## select the colors
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-1')

Bi-clustering 2: combine smallest eigenvectors

We observe that the last two eigenvalues are very close to each other.

In [ ]:
# Combine the first two eigenvectors to see whether we have better results?
tol = 1e-4

from itertools import combinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol_ev = 0.02
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol_ev]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("#Flip angles:", len(flip_angles))
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = []
best_partition = []

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost-tol:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif abs(cost-best_cost) < tol:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost/fac)
print("# Best partitions:", len(best_partition))

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-2')

###### cut = 5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z2-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'
# Set tick positions (match number of labels)
# xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
# yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
# plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
# plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 0.01
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

Bi-clustering 1: choose the smallest eigenvalue:

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-1')

Bi-clustering 2: combine smallest eigenvectors

Observation: the first two eigenvalues are close to each other, compared with others

In [ ]:
# Combine the first two eigenvectors to see whether we have better results?
tol = 1e-4

from itertools import combinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol_ev = 0.031
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol_ev]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("#Flip angles:", len(flip_angles))
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = []
best_partition = []

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost-tol:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif abs(cost-best_cost) < tol:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost/fac)
print("# Best partitions:", len(best_partition))

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-2')

###### cut = 1.5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 1.5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z2-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'
# Set tick positions (match number of labels)
# xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
# yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
# plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
# plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=0.80,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

#### Zone III: MY

In [ ]:
# --- Step 1: select a region ---
xlim = [1695., 1760.]
ylim = [4000., 4230.]
zlim = [1640., 1740.]

In [ ]:
nodes_sel, regs_sum = select_vertices_range(df_nodes, xlim, ylim, zlim)
print("regions:", regs_sum[0])
print("counts:", regs_sum[1])

This is correct, we get extrac edges from other cc for MY-sat

In [ ]:
# --- Step 2: select edges touching at least one selected node ---
df_edges_sel = df_edges[df_edges["node1id"].isin(nodes_sel) | df_edges["node2id"].isin(nodes_sel)]

# --- Step 3: update node set with all incident nodes ---
nodes_sel2 = set(df_edges_sel["node1id"]).union(set(df_edges_sel["node2id"]))
df_nodes_sel2 = df_nodes[df_nodes["id"].isin(nodes_sel2)]

print(f"Initial selected nodes: {len(nodes_sel)}")
print(f"Selected edges: {len(df_edges_sel)}")
print(f"Updated node set: {len(nodes_sel2)}")

In [ ]:
# create a node label dictionary
n2i_dict = {id:i for i,id in enumerate(df_nodes_sel2['id'].values)}
print("#nodes in dictionary", len(n2i_dict))

In [ ]:
# Region labels and colors
region_categories = sorted(regs_sum[0])
print(region_categories)

# region_color_map = {
#     region_categories[0]: "red",
#     region_categories[1]: "blue",
#     region_categories[2]: "black",
# }
region_color_map = {
    region_categories[0]: '#BCBD22',
    region_categories[1]: '#9467BD',
    region_categories[2]: '#F58518', #'#8C564B'
}

In [ ]:
# Build a fast lookup for node positions - edges color ~ radius
ns = 2.5
lw = 3.5
xval = 0.8
cmap = 'coolwarm'

pos_dict = df_nodes_sel2.set_index("id")[
    ["pos_x", "pos_y", "pos_z"]
].to_dict("index")

# prepare the color map
r_vals = df_edges_sel["minRadiusAvg"].values
val_min, val_max = r_vals.min(), r_vals.max()
print("max, min abs value in edge vals:", val_max, val_min)

# Normalize + convert colormap
colorscale = matplotlib_to_plotly_colorscale(cmap)
# Colorbar
colorbar_trace = dummy_colorbar_trace(val_min, val_max, colorscale, xval=xval)

# assign colors
norm = mcolors.Normalize(vmin=val_min, vmax=val_max)
cmap_vals = plt.colormaps.get_cmap(cmap)

edge_traces = []
for _, edge in df_edges_sel.iterrows():
    n1, n2 = edge["node1id"], edge["node2id"]

    if (n1 not in pos_dict) or (n2 not in pos_dict):
        continue

    x0, y0, z0 = pos_dict[n1]["pos_x"], pos_dict[n1]["pos_y"], pos_dict[n1]["pos_z"]
    x1, y1, z1 = pos_dict[n2]["pos_x"], pos_dict[n2]["pos_y"], pos_dict[n2]["pos_z"]

    r = edge["minRadiusAvg"]
    # normalized_r = (r - r_min) / (r_max - r_min + 1e-12)
    color = mcolors.to_hex(cmap_vals(norm(r)))
    edge_traces.append(go.Scatter3d(
            x=[x0, x1, None],
            y=[y0, y1, None],
            z=[z0, z1, None],
            mode="lines",
            line=dict(color=color, width=lw),
            hoverinfo="text",
            text=f"Radius: {r:.3f}",
            showlegend=False,
        ))

node_traces = []
for region, color in region_color_map.items():
    df_r = df_nodes_sel2[df_nodes_sel2["region_label"] == region]

    node_traces.append(
        go.Scatter3d(
            x=df_r["pos_x"],
            y=df_r["pos_y"],
            z=df_r["pos_z"],
            mode="markers",
            name=f"Region {region}",
            marker=dict(
                size=ns,
                color=color,
                opacity=0.9,
            ),
            hoverinfo="text",
            text=[
                f"Node {nid}<br>Region: {region}"
                for nid in df_r["id"]
            ],
        )
    )

fig = go.Figure(data=[*node_traces, *edge_traces, colorbar_trace])

axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=True,
        )


fig.update_layout(
    # title="3D Brain Vascular Subnetwork",
    scene=dict(
        xaxis=dict(title="x"),
        yaxis=dict(title="y"),
        zaxis=dict(title="z"),
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    # legend=dict(itemsizing="constant"),
)

fig.show()

In [ ]:
#set an appropriate camera
camera = dict(
    eye=dict(x=1.2, y=.8, z=.6),
    center=dict(x=0, y=0, z=0),
    up=dict(x=0, y=0, z=.1)
)
fig.update_layout(scene_camera=camera)
fig.show()

In [ ]:
pio.write_image(fig, "vessel-z3_view.png", width=700, height=600, scale=2) #, scale=2

In [ ]:
# output the range
full_fig = fig.full_figure_for_development()
print(full_fig.layout.scene.xaxis.range)
print(full_fig.layout.scene.yaxis.range)
print(full_fig.layout.scene.zaxis.range)

##### Edge counts

In [ ]:
### see how #edges change along the radius cuts
col_rad = "minRadiusAvg"
# cuts = np.arange(1., 16., 0.5)
cuts = sorted(df_edges_sel[col_rad].unique()[1:-1])

summary = []
for cut in cuts:
    # summarize the edges below and above
    num_below = len(df_edges_sel[df_edges_sel[col_rad] < cut])
    num_above = len(df_edges_sel[df_edges_sel[col_rad] >= cut])

    summary.append([cut, num_below, num_above])

In [ ]:
# visualize the summary
tk_fz = 14
lb_fz = 16
plt.figure(figsize=(7,3.5))
summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.savefig("vessel-z3-edgrad.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# visualize the summary + cuts
cut_sel = [1.5, 2, 5]

plt.figure(figsize=(7,3.5))
# plot vertical lines
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

summary = np.array(summary)
plt.plot(summary[:, 0], summary[:, 1], '.--', c='b', alpha=0.6)
plt.plot(summary[:, 0], summary[:, 2], '.--', c='r', alpha=0.6)
# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('# edges', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.savefig("vessels-z3-edgradc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Q: how does the edge distance correlate with radius?
res_dist = []
res_radi= []
G2 = build_region_graphs(df_nodes_sel2, df_edges_sel)

for _, row in df_edges_sel.iterrows():
    n1, n2 = row["node1id"], row["node2id"]
    res_radi.append(float(row["minRadiusAvg"]))

    # compute the distance between the two nodes
    pos1 = np.array(G2.nodes[n1]['pos'])
    pos2 = np.array(G2.nodes[n2]['pos'])
    dist = np.linalg.norm(pos1-pos2)
    res_dist.append(dist)

In [ ]:
# pearson correlation
from scipy import stats

corr = stats.pearsonr(res_dist, res_radi)
print("correlation:", corr)

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels-z3-dist-radius.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(5,3.8))
plt.scatter(res_dist, res_radi, marker='.', alpha=.5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Distance')
plt.ylabel('Radius')
plt.savefig("vessels-z3-dist-radius-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Unbalance scores + Centralities

In [ ]:
### See how balance changes with cut
# sort edges
col_rad = "minRadiusAvg"
df_edges_sel = df_edges_sel.sort_values(by=col_rad)

G_sel = build_region_graphs(df_nodes_sel2, df_edges_sel)

## compute the full GLI between each pair of edges in order
Lgli_full = np.zeros((len(df_edges_sel), len(df_edges_sel)))
for i in range(Lgli_full.shape[0]):
    e1 = [df_edges_sel.iloc[i]['node1id'], df_edges_sel.iloc[i]['node2id']]
    e1_pos = [np.array(G_sel.nodes[e1[0]]['pos']), np.array(G_sel.nodes[e1[1]]['pos'])]
    for j in range(i+1, Lgli_full.shape[1]):
        e2 = [df_edges_sel.iloc[j]['node1id'], df_edges_sel.iloc[j]['node2id']]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lgli_full[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G_sel.nodes[e2[0]]['pos']), np.array(G_sel.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lgli_full[i, j] = gli_ij

In [ ]:
### See how *centrality distribution* changes with cut
res_cut = []

res_bal = []
res_cme = []
res_csd = []
res_cax = []
idx = 0
for _, row in df_edges_sel.iterrows():

    # obtain the linking
    Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
    print("Lam_gli shape:", Lam_gli.shape)

    if Lam_gli.shape[1] == 0:
        continue

    # record the cut
    res_cut.append(float(row[col_rad]))
    print("cut:", res_cut[-1])

    idx += 1

    # weigh matrix of the bipartite graph
    W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                      [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

    ## Balance score
    # prepare connected components if any
    Gb = nx.from_numpy_array(W_gli)
    Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
    print("#connected components:", len(Gb_ccs))
    print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

    # for each cc
    res = []
    for idx_g in range(len(Gb_ccs)):
        Gb_cc = Gb_ccs[idx_g]
        if Gb_cc.number_of_nodes() < 3:
            continue
        # get the adjacency
        W_cc = nx.to_numpy_array(Gb_cc)
        deg = np.sum(abs(W_cc), axis=1)

        # symmetric version
        D2_inv = np.diag(1./np.sqrt(deg))
        P_sym = D2_inv.dot(W_cc).dot(D2_inv)
        eigvals = np.linalg.eigvals(P_sym)
        # print("Max eigenval of P:", np.max(eigvals))
        res.append(np.max(eigvals))
    print("balance score:", res)
    # set one balance score for each cut value
    if len(res) == 0:
        res.append(np.nan)
    else:
        res_bal.append(np.mean(res).real)

    ## Centralities
    res = np.sum(abs(W_gli), axis=0)
    res_cme.append(np.mean(res))
    res_csd.append(np.std(res))
    res_cax.append(np.max(res))

1. Unbalance score

In [ ]:
res_cut = np.array(res_cut)
res_bal = np.array(res_bal)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'bal': res_bal})
df_res.to_csv("vessels-z3-balrad.csv", index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-balrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unalance score')
plt.savefig("vessels-z3-unbalrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

# plt.scatter(cut_arr, bal_arr, marker='.', alpha=0.6)
plt.plot(res_cut, 1-res_bal, marker='.', linestyle='dashed', alpha=0.6, c='blueviolet')
plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.savefig("vessels-z3-unbalrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# set one balance score for each cut value
res_cut = np.array(res_cut)
res_cme = np.array(res_cme)
res_csd = np.array(res_csd)
res_cax = np.array(res_cax)

In [ ]:
# save data
df_res = pd.DataFrame({'cut': res_cut, 'cent-mean': res_cme, 'cent-std': res_csd, 'cent-max': res_cax})
df_res.to_csv("vessels-z3-cenrad.csv", index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z1-cenrad.csv", index=False)

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cenrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)

plt.plot(res_cut, res_cme, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.fill_between(res_cut, res_cme-res_csd, res_cme+res_csd, alpha=0.2, color='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cenrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cmxrad-l.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))
for cut in cut_sel:
    plt.axvline(x=cut, linestyle='--', color='k', alpha=0.5)
plt.plot(res_cut, res_cax, marker='.', linestyle='dashed', alpha=0.6, c='orangered')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cmxrad-lc.png", dpi=300, bbox_inches='tight')
plt.show()

###### Null model 1: SER and SRG

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['er', 'dist'][0]
p_dist = True if lab_mod == 'dist' else False
print("probability by distance?", p_dist)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    nodes_null_df, edges_null_df, G_null = generate_spatial_null_model(
        n_nodes=num_nodes, m_edges=num_edges, df_edges_sel=df_edges_sel,
        xyz_ranges=xyz_ranges, directed=False, p_dist=p_dist)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij
    # compute the balance score
    res_cut = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # characterize the balance
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
# plt.savefig("vessels-z3-unbalrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
# df_res.to_csv("vessels-z3-unbalrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)
df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
  # plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cenrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-cenrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cmerad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-cmerad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-caxrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
#  plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-caxrad-sw-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z3-cenrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z3-cenrad-sw-{}-s{}.csv".format(lab_mod, num_sam), index=False)

###### Null model 2: shuffle radius

In [ ]:
# random sample and compute the unbalance score for comparison
num_sam = 10

# choose the model
lab_mod = ['radi'][0]
print("null model", lab_mod)

num_nodes = len(nodes_sel2)
num_edges = len(df_edges_sel)
# Use empirical bounding box for the null positions
xyz_ranges = ((xlim[0], xlim[1]), (ylim[0], ylim[1]), (zlim[0], zlim[1]))

col_rad = "minRadiusAvg"

resall_cut = []
# balance
resall_bal = []
# centrality
resall_cut = []
resall_cme = []
resall_csd = []
resall_cax = []

for s in range(num_sam):
    print("sample:", s)

    ### generate null models
    edges_null_df, G_null = generate_radii_null_model(
        m_edges=num_edges, df_edges_sel=df_edges_sel, df_nodes_sel=df_nodes_sel2, directed=False)

    ### sweep the unbalance score
    # sort edges
    edges_null_df = edges_null_df.sort_values(by=col_rad)

    # compute the full GLI between each pair of edges in order
    Lgli_full = np.zeros((len(edges_null_df), len(edges_null_df)))
    for i in range(Lgli_full.shape[0]):
        e1 = [edges_null_df.iloc[i]['node1id'], edges_null_df.iloc[i]['node2id']]
        e1_pos = [np.array(G_null.nodes[e1[0]]['pos']), np.array(G_null.nodes[e1[1]]['pos'])]
        for j in range(i+1, Lgli_full.shape[1]):
            e2 = [edges_null_df.iloc[j]['node1id'], edges_null_df.iloc[j]['node2id']]
            # check whether they share a vertex
            if e1[0] in e2 or e1[1] in e2:
                Lgli_full[i, j] = 0
            else:
                # record the position
                e2_pos = [np.array(G_null.nodes[e2[0]]['pos']), np.array(G_null.nodes[e2[1]]['pos'])]
                # direct computation of Gauss linking integral
                gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
                Lgli_full[i, j] = gli_ij

    # compute the balance score
    res_cut = []
    res_bal = []
    res_cme = []
    res_csd = []
    res_cax = []
    idx = 0
    for _, row in edges_null_df.iterrows():

        # obtain the linking
        Lam_gli = Lgli_full[:idx+1, :][:, idx+1:]
        print("Lam_gli shape:", Lam_gli.shape)

        if Lam_gli.shape[1] == 0:
            continue

        # record the cut
        res_cut.append(float(row[col_rad]))
        print("cut:", res_cut[-1])

        idx += 1
        # weight matrix of the bipartite graph
        W_gli = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli],
                          [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

        ## Balance
        # prepare connected components if any
        Gb = nx.from_numpy_array(W_gli)
        Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
        print("#connected components:", len(Gb_ccs))
        print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

        # for each cc
        res = []
        for idx_g in range(len(Gb_ccs)):
            Gb_cc = Gb_ccs[idx_g]
            if Gb_cc.number_of_nodes() < 3:
                continue
            # get the adjacency
            W_cc = nx.to_numpy_array(Gb_cc)
            deg = np.sum(abs(W_cc), axis=1)

            # symmetric version
            D2_inv = np.diag(1./np.sqrt(deg))
            P_sym = D2_inv.dot(W_cc).dot(D2_inv)
            eigvals = np.linalg.eigvals(P_sym)
            # print("Max eigenval of P:", np.max(eigvals))
            res.append(np.max(eigvals))
        print("balance score:", res)
        # set one balance score for each cut value
        if len(res) == 0:
            res.append(np.nan)
        else:
            res_bal.append(np.mean(res).real)

        ## Centrality
        # record the distribution of centralities
        res = np.sum(abs(W_gli), axis=0)
        res_cme.append(np.mean(res))
        res_csd.append(np.std(res))
        res_cax.append(np.max(res))

    # store the results
    resall_cut.append(res_cut)
    resall_bal.append(res_bal)
    resall_cme.append(res_cme)
    resall_csd.append(res_csd)
    resall_cax.append(res_cax)

1. Unbalance score

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

bal_arr = np.array(resall_bal).mean(axis=0)
bal_std = np.array(resall_bal).std(axis=0)

In [ ]:
# plot average +- standard deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, 1-bal_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, 1-bal_arr-bal_std, 1-bal_arr+bal_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Balance score')
plt.savefig("vessels-z3-unbalrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'bal': bal_arr, 'bal_std': bal_std})
df_res.to_csv("vessels-z3-unbalrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_bal = pd.read_csv("vessels-z3-balrad.csv")
df_radi = pd.read_csv("vessels-z3-unbalrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Unbalance score')
plt.legend()
plt.savefig("vessels-z3-unbalrad-null2.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
cut_arr = np.array(resall_cut).mean(axis=0)
cut_std = np.array(resall_cut).std(axis=0)
# check the std of cuts
print("Cut std:", min(cut_std), max(cut_std))

cme_arr = np.array(resall_cme).mean(axis=0)
cme_std = np.array(resall_cme).std(axis=0)
csd_arr = np.array(resall_csd).mean(axis=0)
csd_std = np.array(resall_csd).std(axis=0)
cax_arr = np.array(resall_cax).mean(axis=0)
cax_std = np.array(resall_cax).std(axis=0)

In [ ]:
# plot average +- standard deviation: centrality mean & centrality deviation
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-csd_arr, cme_arr+csd_arr, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cenrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-cenrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality mean
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cme_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cme_arr-cme_std, cme_arr+cme_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-cmerad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-cmerad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot average +- standard deviation: centrality max
plt.figure(figsize=(7,3.5))
plt.plot(cut_arr, cax_arr, linestyle='dashed', marker='.', alpha=0.6, c='k')
plt.fill_between(cut_arr, cax_arr-cax_std, cax_arr+cax_std, alpha=0.2, color='grey')
plt.xlabel('Radius cuts')
plt.ylabel('Centrality')
plt.savefig("vessels-z3-caxrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
#  plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-caxrad-null-{}-s{}.png".format(lab_mod, num_sam), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# save the data
df_res = pd.DataFrame({'cut': cut_arr, 'cent-mean-mean': cme_arr, 'cent-mean-std': cme_std,
                       'cent-std-mean': csd_arr, 'cent-std-std': csd_std,
                       'cent-max-mean': cax_arr, 'cent-max-std': cax_std})
df_res.to_csv("vessels-z3-cenrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)
# df_res.to_csv("/content/drive/MyDrive/ensnarl_res/vessels-z3-cenrad-null-{}-s{}.csv".format(lab_mod, num_sam), index=False)

In [ ]:
### comparison
# read in data
df_cen = pd.read_csv("vessels-z3-cenrad.csv")
df_radi = pd.read_csv("vessels-z3-cenrad-null-radi-s10.csv")

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Mean')
plt.legend()
plt.savefig("vessels-z3-cmerad-null2.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-cmerad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('Radius cuts')
plt.ylabel('Centrality Max')
plt.legend()
plt.savefig("vessels-z3-caxrad-null2.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-caxrad-null2.png", dpi=300, bbox_inches='tight')
plt.show()

###### Summary

In [ ]:
# # uniform scale
# ylim_unb = [-0.05, 0.63]
# ylim_cme = [0.0005, 45]
# ylim_cax = [0.1, 180]

1. Unbalance scores

In [ ]:
## ALL
df_bal = pd.read_csv("vessels-z3-balrad.csv")
df_radi = pd.read_csv("vessels-z3-unbalrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z3-unbalrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z3-unbalrad-sw-dist-s10.csv")

In [ ]:
# plot together
tk_fz = 14
lb_fz = 16
lg_fz = 12

plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)

In [ ]:
colors = ['r', 'b', 'y', 'c']
# plot together
tk_fz = 14
lb_fz = 16
lg_fz = 12

plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-unbalrad-all.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], 1-df_er['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], 1-df_er['bal']-df_er['bal_std'], 1-df_er['bal']+df_er['bal_std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_unb)
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-unbalrad-all3.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-all3.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['bal']-df_radi['bal_std'], 1-df_radi['bal']+df_radi['bal_std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], 1-df_dist['bal'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], 1-df_dist['bal']-df_dist['bal_std'], 1-df_dist['bal']+df_dist['bal_std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_bal['cut'], 1-df_bal['bal'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('unbalance score', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-unbalrad-null3.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-unbalrad-null3.png", dpi=300, bbox_inches='tight')
plt.show()

2. Centrality

In [ ]:
# read in data
df_cen = pd.read_csv("vessels-z3-cenrad.csv")
df_radi = pd.read_csv("vessels-z3-cenrad-null-radi-s10.csv")
df_er = pd.read_csv("vessels-z3-cenrad-sw-er-s10.csv")
df_dist = pd.read_csv("vessels-z3-cenrad-sw-dist-s10.csv")

In [ ]:
colors = ['orangered', 'b', 'y', 'c']

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 1-df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-cmerad-all.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-cenrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-cmerad-all-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-cenrad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-mean-mean']-df_er['cent-mean-std'],
                 df_er['cent-mean-mean']+df_er['cent-mean-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cme)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-cmerad-all3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-cenrad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-mean-mean']-df_radi['cent-mean-std'],
                 df_radi['cent-mean-mean']+df_radi['cent-mean-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-mean-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-mean-mean']-df_dist['cent-mean-std'],
                 df_dist['cent-mean-mean']+df_dist['cent-mean-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-mean'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality mean', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-cmerad-null3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-cenrad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], 1-df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], 1-df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 1-df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

# plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-caxrad-all.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-caxrad-all.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-caxrad-all-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-caxrad-all-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - ER
plt.plot(df_er['cut'], df_er['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[2], label='SER')
plt.fill_between(df_er['cut'], df_er['cent-max-mean']-df_er['cent-max-std'],
                 df_er['cent-max-mean']+df_er['cent-max-std'], alpha=0.2, color=colors[2])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.ylim(ylim_cax)
plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False, loc='lower center')
plt.savefig("vessels-z3-caxrad-all3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-caxrad-all3-log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot together
plt.figure(figsize=(7,3.5))

# null model - radius
plt.plot(df_radi['cut'], df_radi['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[3], label='Rad')
plt.fill_between(df_radi['cut'], df_radi['cent-max-mean']-df_radi['cent-max-std'],
                 df_radi['cent-max-mean']+df_radi['cent-max-std'], alpha=0.2, color=colors[3])

# null model - dist
plt.plot(df_dist['cut'], df_dist['cent-max-mean'], linestyle='dashed', marker='.', alpha=0.3, c=colors[1], label='SRG')
plt.fill_between(df_dist['cut'], df_dist['cent-max-mean']-df_dist['cent-max-std'],
                 df_dist['cent-max-mean']+df_dist['cent-max-std'], alpha=0.2, color=colors[1])

# actual value
plt.plot(df_cen['cut'], df_cen['cent-max'], linestyle='dashed', marker='.', alpha=0.5, c=colors[0], label='real')

plt.yscale('log')
plt.xlabel('radius cuts', fontsize=lb_fz)
plt.ylabel('centrality max', fontsize=lb_fz)
plt.tick_params(axis='both', labelsize=tk_fz)
plt.legend(fontsize=lg_fz, frameon=False)
plt.savefig("vessels-z3-caxrad-null3-log.png", dpi=300, bbox_inches='tight')
# plt.savefig("/content/drive/MyDrive/ensnarl_res/vessels-z3-caxrad-null3-log.png", dpi=300, bbox_inches='tight')
plt.show()

##### Single cut

In [ ]:
# Is the graph connected?
G_sel2 = build_region_graphs(df_nodes_sel2, df_edges_sel)
component_summary = summarize_cc_topk(G_sel2, k=10)

In [ ]:
figsize_g = [850, 500]
figsize_c = [850, 500]
cbar_len = 0.6
xval = 0.90

###### cut = 2

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 2.

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z3-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize
figsize_g = [850, 500]

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'
# Set tick positions (match number of labels)
# xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
# yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
# plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
# plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# set the same range (from exploratory analysis)
cmin = 0.0067179
cmax = 3.5062 # 5.9063 -- there is only one edge of this value in the case of c=5
crange = [cmin, cmax]

In [ ]:
### linking centrality ###
figsize_c = [850, 500]
cbar_len = 0.6
xval = 0.90

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=xval,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange, cbar_len=cbar_len)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

Bi-clustering 1: choose the smallest eigenvalue:

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
## select the colors
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-1')

In [ ]:
# if we only want to show the most negative ones:
plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, [], [],
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-str-1')

Bi-clustering 2: combine smallest eigenvectors

We observe that the last two eigenvalues are very close to each other.

In [ ]:
# Combine the first two eigenvectors to see whether we have better results?
tol = 1e-4

from itertools import combinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol_ev = 0.02
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol_ev]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("#Flip angles:", len(flip_angles))
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = []
best_partition = []

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost-tol:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif abs(cost-best_cost) < tol:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost/fac)
print("# Best partitions:", len(best_partition))

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
alphas = [0.3, 0.5, 1.]
color1 = "rgba(0, 0, 139, {})"
color2 = "rgba(139, 0, 0, {})"

colorset = [[color1.format(a), color2.format(a)] for a in alphas]
# change the middle color
alpha = 0.9
colorset[1] = ["rgba(33, 89, 209,{})".format(alpha), "rgba(196, 29, 29,{})".format(alpha)]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-2')

In [ ]:
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, [], [],
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-str-2')

###### cut = 5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z3-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'
# Set tick positions (match number of labels)
# xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
# yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
# plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
# plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###

# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=xval,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange, cbar_len=cbar_len)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 0.01
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

In [ ]:
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, [], [],
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-str')

###### cut = 1.5

In [ ]:
# Case 1: when # above and below are about the same
ns = 2.
lw = 3.5
cut = 1.5

## Visualization
# --- 1. Select edges for G1 (minRadiusAvg < cut) ---
edges_G1 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] < cut]

# --- 2. Select remaining edges for G2 (minRadiusAvg >= cut OR missing) ---
edges_G2 = [(u, v, data) for u, v, data in G_sel2.edges(data=True) if data["minRadiusAvg"] >= cut]

# print("#edges:", len(edges_G1), len(edges_G2))

# --- 3. Build subgraphs while preserving attributes ---
G1 = nx.Graph()
G1.add_nodes_from(G_sel2.nodes(data=True))
G1.add_edges_from(edges_G1)

G2 = nx.Graph()
G2.add_nodes_from(G_sel2.nodes(data=True))
G2.add_edges_from(edges_G2)

# --- 4. Remove isolated nodes ---
# G1.remove_nodes_from(list(nx.isolates(G1)))
# G2.remove_nodes_from(list(nx.isolates(G2)))

pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')
lab_G = 'vessel-z3-c{}'.format(cut)

In [ ]:
# edge colors
alpha = 0.9
row_c = "rgba(33, 89, 209,{})".format(alpha)
col_c = "rgba(196, 29, 29,{})".format(alpha)
# tick title sizes
ax_fz = 30
# figsize

plot_networkx_12nolabel(G1, G2, pos1, pos2, size=ns, width=lw, row_c=row_c, col_c=col_c,
                        camera=camera, savefig=True, figname=lab_G+'-unic-G', figsize=figsize_g, ax_fz=ax_fz) #row_c='#2159d1', col_c='#c41d1d'

In [ ]:
## GLI analysis
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # check whether they share a vertex
        if e1[0] in e2 or e1[1] in e2:
            Lam_gli[i, j] = 0
        else:
            # record the position
            e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
            # direct computation of Gauss linking integral
            gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
            Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
cmap_max = 0.1
val_max = np.abs(Lam_gli).max()
val_min = np.abs(Lam_gli[Lam_gli != 0]).min()
num_0s = np.sum(Lam_gli == 0)
print("edges in 1 and 2:", Lam_gli.shape)
print("#0 values in Lam_gli:", num_0s)
print("max abs value in Lam_gli:", val_max, "min abs value:", val_min)

In [ ]:
cmap_max = min([cmap_max, val_max])
print("max in cmap:", cmap_max)

plt.figure(figsize=(6, 3.5))
# plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
plt.imshow(Lam_gli, cmap='seismic', vmax=cmap_max, vmin=-cmap_max) #, aspect='auto'
# Set tick positions (match number of labels)
# xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
# yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
# plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
# plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli-max{}.png".format(lab_G, cmap_max), dpi=300, bbox_inches='tight')
plt.show()

We see that there are very small values in the GLI operator, should we treat them as zero (i.e., no contribution) or as a contribution?
- Let's maintain the values and explore the patterns there.

In [ ]:
### linking centrality ###
# rows (edges1) and columns (edges2) that are not all zeros
row_sum = np.sum(abs(Lam_gli), axis=1)
col_sum = np.sum(abs(Lam_gli), axis=0)

print("node size:", ns, "line width:", lw)

cmap = 'plasma'
pos1 = nx.get_node_attributes(G1, 'pos')
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sum, col_sum, cmap=cmap, xval=xval,
                     ns=ns, lw=lw, camera=camera, savefig=True, figname=lab_G+'-unic-G-cent',
                     figsize=figsize_c, val_range=crange, cbar_len=cbar_len)

In [ ]:
# bipartite graph clustering
### Construct the bipartite signed graph
fac = 100 # scale up the values, for numerical reasons
Lam_gfac = Lam_gli * fac
# labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
# labels_p2 = ['({},{})'.format(e[0], e[1]) for e in edges2]

# Gb = build_sibigraph(Lam_gtol, labels_p1, labels_p2)

# use index instead
labels_p1 = [i for i in range(len(edges1))]
labels_p2 = [i+len(edges1) for i in range(len(edges2))]

Gb = build_sibigraph(Lam_gfac, labels_p1, labels_p2)
print("#nodes:", Gb.number_of_nodes(), "#edges:", Gb.number_of_edges())

# prepare connected components if any
Gb_ccs = [Gb.subgraph(c) for c in nx.connected_components(Gb)]
print("#connected components:", len(Gb_ccs))
print("sizes:", [(g.number_of_nodes(), g.number_of_edges()) for g in Gb_ccs])

In [ ]:
### Following tasks inside each cc ###
idx_g = 0

Gb_cc = Gb_ccs[idx_g]

# get the adjacency
nodes_cc = sorted(Gb_cc.nodes())
W_cc = nx.to_numpy_array(Gb_cc, nodelist=nodes_cc)

# get edges lists
edges1_cc = [edges1[i] for i in nodes_cc if i < len(edges1)]
edges2_cc = [edges2[i-len(edges1)] for i in nodes_cc if i >= len(edges1)]
print("two parts:", (len(edges1_cc), len(edges2_cc)))


# check the balance
deg = np.sum(abs(W_cc), axis=1)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W_cc).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# Laplacian
D = np.diag(deg)
L = D - W_cc
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals)/fac)

# normalized Laplacian
L_sym = np.eye(W_cc.shape[0]) - P_sym
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-2
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # default setting where all in the 1-group
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost/fac)
# print("Best partition:", best_partition)

In [ ]:
### Post processing ###
# apply the optimal orientation
s = best_partition[0]
S = np.diag(s)
W_s = S.dot(W_cc.dot(S))
B_s = W_s[:len(edges1_cc), :][:, -len(edges2_cc):]

# Distribution of the linking values
plt.hist((B_s/fac).flatten(), bins=100)
plt.yscale('log')
plt.show()

In [ ]:
# select strong negative signals
negr, negc = np.where(B_s < -0.1*fac)
print("#very negative edges:", len(negr))

# record
res = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# medium
negr, negc = np.where(B_s < -0.05*fac)
print("#medium negative edges:", len(negr))

# record
res2 = [(edges1_cc[negr[i]], edges2_cc[negc[i]]) for i in range(len(negr))]
res2_idx = [((n2i_dict[e1[0]], n2i_dict[e1[1]]), (n2i_dict[e2[0]], n2i_dict[e2[1]])) for e1,e2 in res]

# visualization
edges1_str = [epair[0] for epair in res]
edges2_str = [epair[1] for epair in res]

edges1_med = [epair[0] for epair in res2]
edges2_med = [epair[1] for epair in res2]

In [ ]:
lab_fac = '-f{}'.format(fac)
# ns = 3.5
# lw = 5
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, edges1_med, edges2_med,
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m')

In [ ]:
print("node size:", ns, "line width:", lw)

plot_networkx_2colore_sel(G_sel2, G1, G2, edges1_str, edges2_str, [], [],
                          ns=ns, lw=lw, camera=camera, savefig=True, colorset=colorset,
                          figsize=figsize_g, figname=lab_G+lab_fac+'-unic-G-signed-l2m-str')